 15.1 — Setup

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import re

PROJECT_DIR = Path(
    r"C:\Users\acer\Desktop\ProgettoTesi"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "risultati"
)

TURN_PATH = (
    RESULTS_DIR
    / "profilazione_acustica"
    / "turni_con_label_acustica.csv"
)

print(
    "TURN_PATH esiste:",
    TURN_PATH.exists()
)

turn_df = pd.read_csv(
    TURN_PATH
)

print(
    "\nShape:",
    turn_df.shape
)

print(
    "Pazienti:",
    turn_df["patient_id"].nunique()
)

print(
    "Turni:",
    len(turn_df)
)

print(
    "\nColonne relative alla trascrizione:"
)

text_columns = [
    col
    for col in turn_df.columns
    if (
        "trans" in col.lower()
        or "text" in col.lower()
        or "asr" in col.lower()
    )
]

print(
    text_columns
)

TURN_PATH esiste: True

Shape: (3825, 108)
Pazienti: 90
Turni: 3825

Colonne relative alla trascrizione:
['transcript', 'asr_error']


15.2 — Audit preliminare delle trascrizioni

In [ ]:
print(
    "Colonne:",
    turn_df.columns.tolist()
)

candidate_text_cols = [
    col
    for col in turn_df.columns
    if col.lower() in [
        "transcript",
        "transcription",
        "text",
        "asr_text",
        "whisper_text"
    ]
]

print(
    "\nCandidate colonne testo:",
    candidate_text_cols
)

Colonne: ['patient_id', 'recording_id', 'nome_file', 'canale', 'turn_index', 'turn_id', 'start_seconds', 'end_seconds', 'turn_duration_seconds', 'patient_speakers', 'n_diar_segments_merged', 'preceding_role', 'preceding_speaker', 'response_latency_seconds', 'audio_path_turn', 'peak_amplitude', 'clipping_ratio', 'f0_p5', 'f0_p25', 'f0_median', 'f0_p75', 'f0_p95', 'f0_mean', 'f0_std', 'f0_iqr', 'f0_range_p95_p5', 'voiced_f0_frames', 'rms_mean', 'rms_std', 'rms_p5', 'rms_p95', 'non_silent_duration_seconds', 'speech_activity_ratio', 'speech_activity_percent', 'silence_ratio', 'internal_pause_count', 'internal_pause_total_seconds', 'internal_pause_mean_seconds', 'internal_pause_median_seconds', 'internal_pause_max_seconds', 'internal_pause_rate_per_min', 'speech_to_internal_pause_ratio', 'mfcc_1_mean', 'mfcc_1_std', 'mfcc_2_mean', 'mfcc_2_std', 'mfcc_3_mean', 'mfcc_3_std', 'mfcc_4_mean', 'mfcc_4_std', 'mfcc_5_mean', 'mfcc_5_std', 'mfcc_6_mean', 'mfcc_6_std', 'mfcc_7_mean', 'mfcc_7_std', 'mf

15.3 — Audit delle trascrizioni

In [3]:
audit_df = turn_df.copy()

# Pulizia minima del testo
# NON stiamo ancora classificando nulla
audit_df["transcript_clean"] = (
    audit_df["transcript"]
    .fillna("")
    .astype(str)
    .str.strip()
)

audit_df["has_transcript"] = (
    audit_df["transcript_clean"]
    .str.len()
    .gt(0)
)

# Numero parole ricalcolato direttamente dal transcript
audit_df["transcript_word_count"] = (
    audit_df["transcript_clean"]
    .str.split()
    .str.len()
)

# Copertura generale
n_total = len(audit_df)
n_with_text = int(
    audit_df["has_transcript"].sum()
)
n_without_text = (
    n_total - n_with_text
)
print("Turni totali:", n_total)
print(
    "Turni con transcript:",
    n_with_text
)
print(
    "Turni senza transcript:",
    n_without_text
)
print(
    "Copertura transcript:",
    round(
        100 * n_with_text / n_total,
        2
    ),
    "%"
)

# Pazienti
patients_with_text = (
    audit_df.loc[
        audit_df["has_transcript"],
        "patient_id"
    ]
    .nunique()
)
print(
    "\nPazienti con almeno un transcript:",
    patients_with_text
)

patients_without_text = sorted(
    set(audit_df["patient_id"].unique())
    -
    set(
        audit_df.loc[
            audit_df["has_transcript"],
            "patient_id"
        ]
    )
)

print(
    "Pazienti completamente senza transcript:",
    patients_without_text
)

# Lunghezza delle trascrizioni utilizzabili
usable_text = audit_df[
    audit_df["has_transcript"]
].copy()

print(
    "\nStatistiche numero parole per turno:"
)

display(
    usable_text[
        "transcript_word_count"
    ]
    .describe()
    .to_frame()
)

# Distribuzione per fasce
usable_text["text_length_class"] = pd.cut(
    usable_text["transcript_word_count"],
    bins=[
        0,
        2,
        5,
        10,
        20,
        50,
        np.inf
    ],
    labels=[
        "1-2",
        "3-5",
        "6-10",
        "11-20",
        "21-50",
        ">50"
    ]
)

print(
    "\nDistribuzione lunghezza transcript:"
)

display(
    usable_text[
        "text_length_class"
    ]
    .value_counts(sort=False)
    .rename("n_turns")
    .to_frame()
)

Turni totali: 3825
Turni con transcript: 3710
Turni senza transcript: 115
Copertura transcript: 96.99 %

Pazienti con almeno un transcript: 90
Pazienti completamente senza transcript: []

Statistiche numero parole per turno:


,transcript_word_count
count,3710.000000
mean,6.677898
std,8.635610
min,1.000000
25%,2.000000
50%,4.000000
75%,8.000000
max,157.000000



Distribuzione lunghezza transcript:


,n_turns
text_length_class,
1-2,1124
3-5,1158
6-10,819
11-20,414
21-50,180
>50,15


15.4 — Campione reale delle trascrizioni

In [4]:
sample_transcripts = (
    usable_text[
        [
            "patient_id",
            "recording_id",
            "turn_id",
            "turn_duration_seconds",
            "transcript_word_count",
            "transcript"
        ]
    ]
    .sample(
        n=min(30, len(usable_text)),
        random_state=42
    )
    .sort_values(
        [
            "patient_id",
            "turn_id"
        ]
    )
    .reset_index(drop=True)
)

pd.set_option(
    "display.max_colwidth",
    300
)

display(
    sample_transcripts
)

,patient_id,recording_id,turn_id,turn_duration_seconds,transcript_word_count,transcript
0,21,ID21.StudioRuggi,ID21.StudioRuggi__patient_turn_0023,3.156,4,allora basta zona dorsale
1,22,ID22.StudioRuggi,ID22.StudioRuggi__patient_turn_0004,0.878,2,"Ah, però..."
2,22,ID22.StudioRuggi,ID22.StudioRuggi__patient_turn_0021,3.021,8,"e se non lo si chiama, lo rischia."
3,27,ID27.SStudioRuggi,ID27.SStudioRuggi__patient_turn_0032,3.881,4,"e sopra, sopra, dentro"
4,29,ID29.StudioRuggi,ID29.StudioRuggi__patient_turn_0002,0.860,1,impiegata
5,31,ID31.StudioRuggi (1)-001,ID31.StudioRuggi (1)-001__patient_turn_0006,1.265,1,42
6,32,ID32.StudioRuggi-005,ID32.StudioRuggi-005__patient_turn_0042,1.992,2,e... appungente.
7,34,ID34.StudioRuggi-002,ID34.StudioRuggi-002__patient_turn_0059,4.944,11,"mi muovo tutta, proprio quello che è quando non posso muovere."
8,37,ID37.Studio-Ruggi-014,ID37.Studio-Ruggi-014__patient_turn_0011,3.375,5,"Come erano? Ma tranquillamente, no?"
9,43,ID43.StudioRuggi,ID43.StudioRuggi__patient_turn_0030,1.299,4,e d'arcollo in giù


15.4b — Turni con trascrizioni più lunghe

In [5]:
long_transcripts = (
    usable_text[
        [
            "patient_id",
            "recording_id",
            "turn_id",
            "turn_duration_seconds",
            "transcript_word_count",
            "transcript"
        ]
    ]
    .sort_values(
        "transcript_word_count",
        ascending=False
    )
    .head(15)
    .reset_index(drop=True)
)


display(
    long_transcripts
)

,patient_id,recording_id,turn_id,turn_duration_seconds,transcript_word_count,transcript
0,59,ID59.StudioRuggi,ID59.StudioRuggi__patient_turn_0024,52.177,157,"Certo, lo studio e la lettura e studiare autoidatta sempre da quando sono... Se le dico questa cosa forse mi arrestano ancora. Io non avevo i genitori estruiti, io ho rubato un libro a 7 anni a scuola dal maestro e questo maestro aveva l'intuita che io l'ho presso nel sé, io ve lo avevo. E vi co..."
1,59,ID59.StudioRuggi,ID59.StudioRuggi__patient_turn_0047,57.881,154,"nella vita come dolma e come mamma, capite questo? che è il bello e guardarsi negli occhi, prendere qua il bambino, ma a braccio e guardarsi nell'occhi la pacca non serve a niente, mia mamma con la scopice, ma come c'è chi a pato? il mio padre che io rivolgo la figlia di mio padre, quella è mia ..."
2,59,ID59.StudioRuggi,ID59.StudioRuggi__patient_turn_0086,40.635,99,"non mi pensi che ha le parole, sarebbe quello di un tonfo che ti senti che tu non puoi essere impotente di sentire questo tonfo di oppressione, che poi tu lo cataloghi insieme e magari psicologicamente vorresti farlo diluire, però magari diventa un misto tra una cosa reale della coppa e della co..."
3,59,ID59.StudioRuggi,ID59.StudioRuggi__patient_turn_0032,34.306,99,"Ma fai, questa è la mia labia e la fribizia della vita, qualcosa se ne vuoi curare un bambino. Se nessuno te le dà, la fortuna non te le dà, che non era povera, era proprio povera. E io ho capito però che l'input doveva di ottenere qualche cosa che tu non hai, però tu sapi, quando puoi riuscire ..."
4,4,ID4.StudioRuggi,ID4.StudioRuggi__patient_turn_0018,30.459,83,"L'ultima volta che l'ho fatta la trità, me la ricordo molto bene, era maggio dell'anno scurse, è passato un anno e poi dopo sono venuta quando ho fatto un folla a passione, da qui non era emerso niente, poi è andato al nonarmi e emerso tutto e quindi da allora io sono in chemio, dall'8 agosto e ..."
5,59,ID59.StudioRuggi,ID59.StudioRuggi__patient_turn_0058,34.914,79,"Ha sempre quella di prima. È la persona che mi sa da fare stare bene. In questo periodo sono soprattutto il mio unibotino audistico. Io penso di essere utile come feedback. Penso di fare un bene a lui, soprattutto perché c'è una indiferenza nella società. Quindi quelle che sono già autosufficie,..."
6,59,ID59.StudioRuggi,ID59.StudioRuggi__patient_turn_0049,27.067,75,"che il computer è qua, una volta che tu metti una cosa, una parola inglese straniera, il gioggiere, quella sta là, se l'hai messa bene tu c'hai pensato, quando la push, push quell'esce, perché i miei alunni poi le dicevo, allora quando è studiata non ti si conto come si dice sta fra, perché non ..."
7,112,ID112.StudioRuggi,ID112.StudioRuggi__patient_turn_0036,20.351,73,"No, non mi sento, in anzi perché le penne ne ho solo io e c'è potesse le forte soprattutto per i miei figli Quindi famigliare, in ca yeni l'avevo scusato No no no, per te ho successo, io che sono permissionata No voglio, se no Ma quando mi hai chiesto del dolore, c'hai messo il dolore del Filgas..."
8,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006__patient_turn_0017,26.257,68,"Mi ricordo di fare scenate pure brutte, perché mi sono trovato in mal tempo, mi ha buttato fuori. Ci vi siete in modo di morire due o tre volte, perché in competenza, ecco, perché mi sono abendurato dalla sera alla mattina, però troppo veloce. Come rai a barcchette mi sa fare i dettagli senza es..."
9,59,ID59.StudioRuggi,ID59.StudioRuggi__patient_turn_0012,24.773,67,"erano delle bambine prodigiosi e speciali che hanno automa. Una mamma come Ticcia, quindi ho fatto la mia esperienza su di loro e loro da grande hanno capito che cosa è servito questa mia greia. Questo ho fatto di input, di fargli capire che ci sta già il bilbismo quando si nasce, orpano, di ter..."


15.4c — Ricerca esplorativa di termini legati al dolore
 - NON è ancora il filtro definitivo


In [6]:
exploratory_terms = [
    "dolore",
    "male",
    "dolor",
    "bruci",
    "brucia",
    "bruciore",
    "fitta",
    "fitte",
    "pung",
    "scossa",
    "scosse",
    "formicol",
    "intorpid",
    "fastidio",
    "soffr",
    "peggior",
    "miglior"
]


pattern_exploratory = (
    "|".join(
        re.escape(term)
        for term in exploratory_terms
    )
)


exploratory_matches = (
    usable_text[
        usable_text[
            "transcript_clean"
        ]
        .str.lower()
        .str.contains(
            pattern_exploratory,
            regex=True,
            na=False
        )
    ]
    [
        [
            "patient_id",
            "recording_id",
            "turn_id",
            "transcript_word_count",
            "transcript"
        ]
    ]
    .copy()
)


print(
    "Turni con almeno un termine esplorativo:",
    len(exploratory_matches)
)

print(
    "Pazienti rappresentati:",
    exploratory_matches[
        "patient_id"
    ].nunique()
)


display(
    exploratory_matches
    .sample(
        n=min(
            40,
            len(exploratory_matches)
        ),
        random_state=42
    )
    .sort_values(
        [
            "patient_id",
            "turn_id"
        ]
    )
    .reset_index(drop=True)
)

Turni con almeno un termine esplorativo: 317
Pazienti rappresentati: 78


,patient_id,recording_id,turn_id,transcript_word_count,transcript
0,3,ID3.StudioRuggi,ID3.StudioRuggi__patient_turn_0022,12,"i dolori alla schiena e alle gambe, tutte e due le gambe."
1,4,ID4.StudioRuggi,ID4.StudioRuggi__patient_turn_0007,7,Qual è il tipo di doloriscusa specificato?
2,10,ID10.StudioRuggi,ID10.StudioRuggi__patient_turn_0022,24,"forse questo, perché poi oltre questo dolore, è qua che ho avuto degli interventi chirurgici, ma non mi sembra di aver sofferto questo dolore."
3,14,ID14.StudioRuggi,ID14.StudioRuggi__patient_turn_0026,49,"il dolore, le mani, i piedi, se mi fanno male le unchie delle mani, spiazionano il modo la sera e il dolore che non sopporto che devo strivere, devo fare. I piedi, quando sono senza scarpe, che cammino per terre, per te le cose si compiassero ancora di più."
4,20,ID20.StudioRuggi,ID20.StudioRuggi__patient_turn_0015,19,"un avvenimento che accade, uno ne prende atto e cerca di trovare le soluzioni per vivere nel modo migliore."
5,21,ID21.StudioRuggi,ID21.StudioRuggi__patient_turn_0011,29,"mi ha dato un input più, cioè proprio all'ennesima potenza, in fact anche quando il dolore cerco di comunque da averla vicino perché mi dà un po' di energia."
6,21,ID21.StudioRuggi,ID21.StudioRuggi__patient_turn_0024,14,allora il dolore è più forte è quando mi è venuto la prima crisi
7,26,ID26.StudioRuggi,ID26.StudioRuggi__patient_turn_0037,5,Che tipo di dolore era?
8,30,ID30.StudioRuggi,ID30.StudioRuggi__patient_turn_0043,44,"No, adesso non soffro, semplicemente dolore alla schiena, ma per quando riguarda il tumore, toccurando ancora con una sienga molto costosa di ormoni, è avuto il suo buro effetto, così come pure per insumula la radioterapia, però la radioterapia è prodotta anche in armi."
9,37,ID37.Studio-Ruggi-014,ID37.Studio-Ruggi-014__patient_turn_0052,11,"Ma il dolore ha si avuto proprio non niente, il dolore..."


15.5 — Categorie lessicali pain-related
- PRIMA VERSIONE AD ALTA PRECISIONE

In [7]:
import re
import unicodedata

# Normalizzazione del transcript
def normalize_transcript(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    # spazi multipli
    text = re.sub(
        r"\s+",
        " ",
        text
    )
    return text

audit_df["transcript_norm"] = (
    audit_df["transcript"]
    .apply(normalize_transcript)
)

# CATEGORIA 1 — Riferimento esplicito al dolore
# Uso stem per tollerare anche piccoli errori ASR: dolore, dolori, doloroso, dolorisissimo, doloro...

PATTERN_EXPLICIT_PAIN = re.compile(
    r"\b\w*dolor\w*\b"
    r"|\bsoffr\w*\b"
    r"|\bfastid\w*\b",
    flags=re.IGNORECASE
)

# CATEGORIA 2 — Descrittori sensoriali compatibili col dolore
PATTERN_SENSORY = re.compile(
    r"\b\w*bruci\w*\b"
    r"|\bformicol\w*\b"
    r"|\bintorpid\w*\b"
    r"|\btorpore\b"
    r"|\baddorment\w*\b"
    r"|\b\w*pung\w*\b"
    r"|\bfitt\w*\b"
    r"|\btrafitt\w*\b"
    r"|\bscoss\w*\b"
    r"|\belettric\w*\b"
    r"|\blancin\w*\b"
    r"|\bpuls\w*\b"
    r"|\bcramp\w*\b"
    r"|\boppress\w*\b",
    flags=re.IGNORECASE
)

# CATEGORIA 3 — "male" solo con contesto
# NON uso semplicemente la parola "male", perché sarebbe troppo ambigua.

PATTERN_MALE_CONTEXT = re.compile(
    r"\bmi\s+fa\s+male\b"
    r"|\bmi\s+fanno\s+male\b"
    r"|\bfa\s+male\b"
    r"|\bfanno\s+male\b"
    r"|\btroppo\s+male\b"
    r"|\bmale\s+(alla|alle|al|ai|allo|agli|"
    r"sulla|sulle|sul|sui|nella|nelle|nel|nei|"
    r"qui|lì|li)\b",
    flags=re.IGNORECASE
)

# Applicazione delle tre categorie
audit_df["pain_explicit"] = (
    audit_df["transcript_norm"]
    .str.contains(
        PATTERN_EXPLICIT_PAIN,
        regex=True,
        na=False
    )
)
audit_df["pain_sensory"] = (
    audit_df["transcript_norm"]
    .str.contains(
        PATTERN_SENSORY,
        regex=True,
        na=False
    )
)
audit_df["pain_male_context"] = (
    audit_df["transcript_norm"]
    .str.contains(
        PATTERN_MALE_CONTEXT,
        regex=True,
        na=False
    )
)

# Candidate pain-related
audit_df["pain_candidate"] = (
    audit_df[
        [
            "pain_explicit",
            "pain_sensory",
            "pain_male_context"
        ]
    ]
    .any(axis=1)
)

# Flag diagnostici — NON comportano ancora esclusione
# Possibile dolore emotivo / psicologico
PATTERN_EMOTIONAL = re.compile(
    r"dolore\s+(interiore|emotivo|psicologico)"
    r"|sofferenza\s+(emotiva|psicologica)"
    r"|dolore\s+dell[' ]anima",
    flags=re.IGNORECASE
)

audit_df["flag_emotional_pain"] = (
    audit_df["transcript_norm"]
    .str.contains(
        PATTERN_EMOTIONAL,
        regex=True,
        na=False
    )
)

# Possibili domande attribuite erroneamente al paziente
PATTERN_QUESTION_LIKE = re.compile(
    r"^\s*(che tipo|qual[e']?\s|quanto|dove|"
    r"come mai|quando)\b",
    flags=re.IGNORECASE
)

audit_df["flag_question_like"] = (
    audit_df["transcript_norm"]
    .str.contains(
        PATTERN_QUESTION_LIKE,
        regex=True,
        na=False
    )
    |
    audit_df["transcript_norm"]
    .str.contains(
        r"\?",
        regex=True,
        na=False
    )
)


print("Filtro preliminare costruito.")

Filtro preliminare costruito.


C:\Users\acer\AppData\Local\Temp\ipykernel_13576\2498663638.py:84: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  audit_df["transcript_norm"]
C:\Users\acer\AppData\Local\Temp\ipykernel_13576\2498663638.py:114: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  audit_df["transcript_norm"]
C:\Users\acer\AppData\Local\Temp\ipykernel_13576\2498663638.py:130: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  audit_df["transcript_norm"]


 15.6 — Distribuzione dei candidate pain-related

In [8]:
print(
    "Turni totali:",
    len(audit_df)
)

print(
    "Turni con transcript:",
    audit_df["has_transcript"].sum()
)

print("\nCATEGORIE:")

for col in [
    "pain_explicit",
    "pain_sensory",
    "pain_male_context",
    "pain_candidate"
]:
    n_turns = int(
        audit_df[col].sum()
    )
    n_patients = (
        audit_df.loc[
            audit_df[col],
            "patient_id"
        ]
        .nunique()
    )
    print(
        f"{col:22s} "
        f"turni={n_turns:4d} | "
        f"pazienti={n_patients:2d}"
    )

print("\nFLAG DA REVISIONARE:")
print(
    "Possibile dolore emotivo:",
    int(
        (
            audit_df["pain_candidate"]
            &
            audit_df["flag_emotional_pain"]
        ).sum()
    )
)

print(
    "Possibili domande:",
    int(
        (
            audit_df["pain_candidate"]
            &
            audit_df["flag_question_like"]
        ).sum()
    )
)


# Pazienti senza alcun candidate pain-related
patients_with_candidate = set(
    audit_df.loc[
        audit_df["pain_candidate"],
        "patient_id"
    ]
)
all_patients = set(
    audit_df["patient_id"]
)
patients_without_candidate = sorted(
    all_patients
    -
    patients_with_candidate
)

print(
    "\nPazienti con almeno un candidate:",
    len(patients_with_candidate)
)

print(
    "Pazienti senza candidate:",
    len(patients_without_candidate)
)

print(
    "ID senza candidate:",
    patients_without_candidate
)

Turni totali: 3825
Turni con transcript: 3710

CATEGORIE:
pain_explicit          turni= 266 | pazienti=75
pain_sensory           turni=  37 | pazienti=23
pain_male_context      turni=  16 | pazienti=11
pain_candidate         turni= 299 | pazienti=78

FLAG DA REVISIONARE:
Possibile dolore emotivo: 1
Possibili domande: 18

Pazienti con almeno un candidate: 78
Pazienti senza candidate: 12
ID senza candidate: [16, 18, 27, 36, 47, 50, 56, 71, 90, 93, 116, 145]


15.7 — Revisione manuale dei candidate

In [9]:
candidate_review = (
    audit_df[
        audit_df["pain_candidate"]
    ]
    [
        [
            "patient_id",
            "turn_id",
            "transcript_word_count",
            "pain_explicit",
            "pain_sensory",
            "pain_male_context",
            "flag_emotional_pain",
            "flag_question_like",
            "transcript"
        ]
    ]
    .sample(
        n=min(
            40,
            int(audit_df["pain_candidate"].sum())
        ),
        random_state=42
    )
    .sort_values(
        [
            "patient_id",
            "turn_id"
        ]
    )
    .reset_index(drop=True)
)

pd.set_option(
    "display.max_colwidth",
    500
)

display(
    candidate_review
)

,patient_id,turn_id,transcript_word_count,pain_explicit,pain_sensory,pain_male_context,flag_emotional_pain,flag_question_like,transcript
0,4,ID4.StudioRuggi__patient_turn_0007,7,True,False,False,False,True,Qual è il tipo di doloriscusa specificato?
1,10,ID10.StudioRuggi__patient_turn_0022,24,True,False,False,False,False,"forse questo, perché poi oltre questo dolore, è qua che ho avuto degli interventi chirurgici, ma non mi sembra di aver sofferto questo dolore."
2,20,ID20.StudioRuggi__patient_turn_0011,37,True,False,False,False,False,"Più che un dolore è un fastidio, un intorpedimento delle mani, quando poi oggi i piedi a terra come se non sentissi la pianta del piede, la sentissi indurita, a volte questo poi mi provoca un fastidio."
3,20,ID20.StudioRuggi__patient_turn_0012,33,True,False,False,False,False,Questo durante la terapia l'anno scorso che avevo dei dolori alle gambe molto forti che non pedivano di dormire durante quella fase poi ha iniziato la terapia che mi è dato il dottore
4,21,ID21.StudioRuggi__patient_turn_0011,29,True,False,False,False,False,"mi ha dato un input più, cioè proprio all'ennesima potenza, in fact anche quando il dolore cerco di comunque da averla vicino perché mi dà un po' di energia."
5,21,ID21.StudioRuggi__patient_turn_0024,14,True,False,False,False,False,allora il dolore è più forte è quando mi è venuto la prima crisi
6,22,ID22.StudioRuggi__patient_turn_0045,13,True,False,False,False,False,"e non è niente, ma è solo un degliano di dolore di uscite."
7,30,ID30.StudioRuggi__patient_turn_0080,25,True,False,False,False,False,"Non riesco a repondere questa domanda, ma è difficile per me descrivere il dolore con un simpolo o un oncetto. È molto difficile per me."
8,32,ID32.StudioRuggi-005__patient_turn_0038,9,False,True,False,False,False,"fortissimo, pungente e non si calmava in nessun modo."
9,34,ID34.StudioRuggi-002__patient_turn_0068,4,True,False,False,False,False,"è dolore, è dolore"


15.8 — Flag più rigoroso per possibili domande del medico

In [10]:
PATTERN_QUESTION_STRICT = re.compile(
    r"^\s*("
    r"qual[e']?\b"
    r"|quali\b"
    r"|che\s+tipo\b"
    r"|quanto\b"
    r"|quanta\b"
    r"|quanti\b"
    r"|quante\b"
    r"|dove\b"
    r")",
    flags=re.IGNORECASE
)

audit_df["flag_question_like_strict"] = (
    audit_df["transcript_norm"]
    .str.contains(
        PATTERN_QUESTION_STRICT,
        regex=True,
        na=False
    )
    &
    audit_df["transcript_norm"]
    .str.contains(
        r"\?",
        regex=True,
        na=False
    )
)

print(
    "Candidate pain-related:",
    int(audit_df["pain_candidate"].sum())
)
print(
    "Candidate con flag domanda STRICT:",
    int(
        (
            audit_df["pain_candidate"]
            &
            audit_df["flag_question_like_strict"]
        ).sum()
    )
)

display(
    audit_df[
        audit_df["pain_candidate"]
        &
        audit_df["flag_question_like_strict"]
    ][
        [
            "patient_id",
            "turn_id",
            "transcript"
        ]
    ]
)

Candidate pain-related: 299
Candidate con flag domanda STRICT: 4


C:\Users\acer\AppData\Local\Temp\ipykernel_13576\2712237386.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  audit_df["transcript_norm"]


,patient_id,turn_id,transcript
856,136,ID136.StudioRuggi__patient_turn_0023,Che tipo di dolore era?
1582,26,ID26.StudioRuggi__patient_turn_0037,Che tipo di dolore era?
2141,4,ID4.StudioRuggi__patient_turn_0007,Qual è il tipo di doloriscusa specificato?
2752,55,ID55.StudioRuggi__patient_turn_0036,Che tipo di dolore era?


15.9a — Campione casuale dei NON-candidate

In [11]:
non_candidate_df = (
    audit_df[
        audit_df["has_transcript"]
        &
        ~audit_df["pain_candidate"]
    ]
    .copy()
)

print(
    "Non-candidate totali:",
    len(non_candidate_df)
)

print(
    "Pazienti rappresentati:",
    non_candidate_df["patient_id"].nunique()
)

non_candidate_random = (
    non_candidate_df[
        [
            "patient_id",
            "recording_id",
            "turn_id",
            "transcript_word_count",
            "transcript"
        ]
    ]
    .sample(
        n=min(40, len(non_candidate_df)),
        random_state=42
    )
    .sort_values(
        [
            "patient_id",
            "turn_id"
        ]
    )
    .reset_index(drop=True)
)

display(
    non_candidate_random
)

Non-candidate totali: 3411
Pazienti rappresentati: 90


,patient_id,recording_id,turn_id,transcript_word_count,transcript
0,10,ID10.StudioRuggi,ID10.StudioRuggi__patient_turn_0006,1,5
1,12,ID12.StudioRuggi,ID12.StudioRuggi__patient_turn_0011,2,e l'intel
2,22,ID22.StudioRuggi,ID22.StudioRuggi__patient_turn_0027,21,"L'alto andivo a guardare qua Una per i bianni, l'altro per il tondice, l'altra per la cazza per il tuo, valessima"
3,27,ID27.SStudioRuggi,ID27.SStudioRuggi__patient_turn_0041,4,non so di scriverlo.
4,27,ID27.SStudioRuggi,ID27.SStudioRuggi__patient_turn_0048,1,oggi
5,30,ID30.StudioRuggi,ID30.StudioRuggi__patient_turn_0005,1,vedo
6,30,ID30.StudioRuggi,ID30.StudioRuggi__patient_turn_0063,5,"alla schiena sulla camberdesta, ginocchio"
7,30,ID30.StudioRuggi,ID30.StudioRuggi__patient_turn_0066,2,è stato
8,34,ID34.StudioRuggi-002,ID34.StudioRuggi-002__patient_turn_0044,11,Io adesso non molto con questa terapia devo dire la verità
9,34,ID34.StudioRuggi-002,ID34.StudioRuggi-002__patient_turn_0078,8,"che è l'uino, ma che è troppo bello."


15.9b — NON-candidate più lunghi

In [12]:
non_candidate_long = (
    non_candidate_df[
        [
            "patient_id",
            "recording_id",
            "turn_id",
            "transcript_word_count",
            "transcript"
        ]
    ]
    .sort_values(
        "transcript_word_count",
        ascending=False
    )
    .head(30)
    .reset_index(drop=True)
)

display(
    non_candidate_long
)

,patient_id,recording_id,turn_id,transcript_word_count,transcript
0,59,ID59.StudioRuggi,ID59.StudioRuggi__patient_turn_0024,157,"Certo, lo studio e la lettura e studiare autoidatta sempre da quando sono... Se le dico questa cosa forse mi arrestano ancora. Io non avevo i genitori estruiti, io ho rubato un libro a 7 anni a scuola dal maestro e questo maestro aveva l'intuita che io l'ho presso nel sé, io ve lo avevo. E vi colto anche che è un libro di racconde dei pinvini del polo norte. Il mio padre lo sa che l'ha fatto, che mi traivano a niente da casa. Ci avevo mettevi documenti dentro, prima io l'ho letto, però io pe..."
1,59,ID59.StudioRuggi,ID59.StudioRuggi__patient_turn_0047,154,"nella vita come dolma e come mamma, capite questo? che è il bello e guardarsi negli occhi, prendere qua il bambino, ma a braccio e guardarsi nell'occhi la pacca non serve a niente, mia mamma con la scopice, ma come c'è chi a pato? il mio padre che io rivolgo la figlia di mio padre, quella è mia mamma mia mamma era disperata perché era solita, sei figli però non rivolgo questa dolcezza che mi ha dato un insegnamento di negli occhi. C'era la rabbia che era stanga della voda, della fatica. Ment..."
2,59,ID59.StudioRuggi,ID59.StudioRuggi__patient_turn_0032,99,"Ma fai, questa è la mia labia e la fribizia della vita, qualcosa se ne vuoi curare un bambino. Se nessuno te le dà, la fortuna non te le dà, che non era povera, era proprio povera. E io ho capito però che l'input doveva di ottenere qualche cosa che tu non hai, però tu sapi, quando puoi riuscire a leggere, non è solo che mi vuoi guardare il suo giovane, questo è un registro di 20 domani, un letter e se lo tenete nel cassetto però io via, non lo vedete per l'uomo un po' chiamate il maestro"
3,4,ID4.StudioRuggi,ID4.StudioRuggi__patient_turn_0018,83,"L'ultima volta che l'ho fatta la trità, me la ricordo molto bene, era maggio dell'anno scurse, è passato un anno e poi dopo sono venuta quando ho fatto un folla a passione, da qui non era emerso niente, poi è andato al nonarmi e emerso tutto e quindi da allora io sono in chemio, dall'8 agosto e quindi via via diciamo la mia capacità di di camminare a passo veloce e siedere tutta, prima a livello chilometrico e poi a livello di velocità."
4,59,ID59.StudioRuggi,ID59.StudioRuggi__patient_turn_0058,79,"Ha sempre quella di prima. È la persona che mi sa da fare stare bene. In questo periodo sono soprattutto il mio unibotino audistico. Io penso di essere utile come feedback. Penso di fare un bene a lui, soprattutto perché c'è una indiferenza nella società. Quindi quelle che sono già autosufficie, già vado a loro, noi non sembriavamo niente, sono loro che devono deviare solo qualche input di mentale. però il bambino autistico è un'altra cosa, appende per l'esperienza."
5,59,ID59.StudioRuggi,ID59.StudioRuggi__patient_turn_0049,75,"che il computer è qua, una volta che tu metti una cosa, una parola inglese straniera, il gioggiere, quella sta là, se l'hai messa bene tu c'hai pensato, quando la push, push quell'esce, perché i miei alunni poi le dicevo, allora quando è studiata non ti si conto come si dice sta fra, perché non c'è pensato, non l'hai inserito, ci devi pensare quando l'ha inserisciato, sta parolando in inglese, sta fratta? Vedi che esce?"
6,1,ID1.StudioRuggi-006,ID1.StudioRuggi-006__patient_turn_0017,68,"Mi ricordo di fare scenate pure brutte, perché mi sono trovato in mal tempo, mi ha buttato fuori. Ci vi siete in modo di morire due o tre volte, perché in competenza, ecco, perché mi sono abendurato dalla sera alla mattina, però troppo veloce. Come rai a barcchette mi sa fare i dettagli senza esperienza e senza rien. Brutto. Però è stato divertente, è stato bello per me."
7,59,ID59.StudioRuggi,ID59.StudioRuggi__patient_turn_0012,67,"erano delle bambine prodigiosi e speciali che hanno automa. Una mamma come Ticcia, quindi ho fatto la mia esperienza su di loro e loro da grande hanno capito che cosa è servito questa mia greia. Questo ho fatto di input, di 

15.10 — Pazienti senza alcun candidate pain-related

In [13]:
missing_candidate_review = (
    audit_df[
        audit_df["patient_id"].isin(
            patients_without_candidate
        )
        &
        audit_df["has_transcript"]
    ]
    .sort_values(
        [
            "patient_id",
            "transcript_word_count"
        ],
        ascending=[
            True,
            False
        ]
    )
    .groupby(
        "patient_id",
        group_keys=False
    )
    .head(5)
    [
        [
            "patient_id",
            "recording_id",
            "turn_id",
            "transcript_word_count",
            "transcript"
        ]
    ]
    .reset_index(drop=True)
)

print(
    "Pazienti esaminati:",
    missing_candidate_review[
        "patient_id"
    ].nunique()
)

display(
    missing_candidate_review
)

Pazienti esaminati: 12


,patient_id,recording_id,turn_id,transcript_word_count,transcript
0,16,ID16.StudioRuggi,ID16.StudioRuggi__patient_turn_0002,25,e fino a poco mi ha vogliato rappresentare il mio cibo e mi ha fatto prendere i pin per memoria perché mi scordo di memoria
1,16,ID16.StudioRuggi(1),ID16.StudioRuggi(1)__patient_turn_0016,22,"a me che non mi piacciono, mi sento con questo cantirema che girano la mattina e cantiremo, cantiremo, che papà sei già"
2,16,ID16.StudioRuggi(1),ID16.StudioRuggi(1)__patient_turn_0018,14,"e non c'è giudice in ricche, e non è un ricche vettore in ricche."
3,16,ID16.StudioRuggi(1),ID16.StudioRuggi(1)__patient_turn_0023,14,"non è potuto dire che non è un appartamento del figlio, non è ormai"
4,16,ID16.StudioRuggi(1),ID16.StudioRuggi(1)__patient_turn_0001,13,"e se non è il fatto di farlo, è il fatto di farlo."
5,18,ID18.StudioRuggi,ID18.StudioRuggi__patient_turn_0004,35,"Prima che io avevo un coltopino, si chiama, un coltopino, poi mi chiamo che è il tuo stoffo, non mi ricordo molto, mi ricordo molto per scoparmi, non mi ricordo, è tutto che a me"
6,18,ID18.StudioRuggi,ID18.StudioRuggi__patient_turn_0005,22,"fatela e travatelo per molti anni. Adesso faccio un utile di presenta a Kurtico Dinapoli, il dottor Michelo Vittor, il dottor General."
7,18,ID18.StudioRuggi,ID18.StudioRuggi__patient_turn_0008,21,"e l'altro è un'altra c'è, la prima è il 13, 78, il 13 febbraio è 78, un'altra è un'altra in pd3"
8,18,ID18.StudioRuggi,ID18.StudioRuggi__patient_turn_0006,12,"Cos'ati? Tanti sei le coline, tre fighe da... Tre donne come ragazzi..."
9,18,ID18.StudioRuggi,ID18.StudioRuggi__patient_turn_0002,6,"Apprenuamo, c'è il giorno di Pasqua."


15.11 — Filtro pain-related contestuale v2


In [14]:
# 1. Descrittori forti osservati anche nei falsi negativi
PATTERN_STRONG_DESCRIPTOR = re.compile(
    r"\bformicol\w*\b"
    r"|\bformigl\w*\b"          # tollera errori ASR tipo "formiglio"
    r"|\btorpore\b"
    r"|\bintorpid\w*\b"
    r"|\baddorment\w*\b"
    r"|\bpung\w*\b"
    r"|\btrafitt\w*\b"
    r"|\bfitt\w*\b"
    r"|\bbruci\w*\b"
    r"|\bscoss\w*\b"
    r"|\belettric\w*\b"
    r"|\blancin\w*\b"
    r"|\bcramp\w*\b"
    r"|\boppress\w*\b"
    r"|\brigid\w*\b"
    r"|\bbloccat\w*\b"
    r"|\bindurit\w*\b"
    r"|\bchiodo\b"
    r"|\bmartello\b",
    flags=re.IGNORECASE
)

# 2. Parti anatomiche non sono sufficienti da sole
PATTERN_BODY = re.compile(
    r"\bschiena\b"
    r"|\bgamb\w*\b"
    r"|\bpied\w*\b"
    r"|\bman\w*\b"
    r"|\bdit\w*\b"
    r"|\bginocchi\w*\b"
    r"|\banca\b"
    r"|\banche\b"
    r"|\bcoscia\b"
    r"|\barto\b"
    r"|\barti\b"
    r"|\bspalla\b"
    r"|\bspalle\b"
    r"|\bcollo\b"
    r"|\bcervical\w*\b"
    r"|\bmandibol\w*\b"
    r"|\btesta\b"
    r"|\bstomaco\b"
    r"|\baddome\b"
    r"|\bpancia\b"
    r"|\bpetto\b"
    r"|\btorace\b"
    r"|\bosso\s+sacro\b",
    flags=re.IGNORECASE
)

# 3. Indicatori di intensità / andamento / limitazione
PATTERN_PAIN_CONTEXT = re.compile(
    r"\bfortissim\w*\b"
    r"|\bmolto\s+forte\b"
    r"|\btroppo\s+forte\b"
    r"|\brealmente\s+forte\b"
    r"|\bnon\s+riesco\s+.*(?:muover|cammin|alz|dorm)\w*\b"
    r"|\bnon\s+riesco\s+pi[uù]\b"
    r"|\bmi\s+sento\s+limitat\w*\b"
    r"|\bdifficolt[aà]\b"
    r"|\bpeggior\w*\b"
    r"|\bmiglior\w*\b"
    r"|\bstare\s+meglio\b"
    r"|\bdura(?:va)?\s+(?:per\s+)?\d+\b"
    r"|\bnon\s+(?:so|saprei|riesco)\s+(?:a\s+)?descriver\w*\b"
    r"|\bdescriver\w*\b",
    flags=re.IGNORECASE
)

# Applicazione
audit_df["pain_strong_descriptor"] = (
    audit_df["transcript_norm"]
    .str.contains(
        PATTERN_STRONG_DESCRIPTOR,
        regex=True,
        na=False
    )
)

audit_df["pain_body_term"] = (
    audit_df["transcript_norm"]
    .str.contains(
        PATTERN_BODY,
        regex=True,
        na=False
    )
)

audit_df["pain_context_term"] = (
    audit_df["transcript_norm"]
    .str.contains(
        PATTERN_PAIN_CONTEXT,
        regex=True,
        na=False
    )
)

# Un termine anatomico entra SOLO se accompagnato da un indicatore contestuale.
audit_df["pain_body_context"] = (
    audit_df["pain_body_term"]
    &
    audit_df["pain_context_term"]
)

# Candidate V2
audit_df["pain_candidate_v2"] = (
    audit_df["pain_candidate"]
    |
    audit_df["pain_strong_descriptor"]
    |
    audit_df["pain_body_context"]
)

# Rimuovo SOLO le quattro domande molto chiaramente attribuibili al medico o voce registrata
audit_df["pain_candidate_v2_clean"] = (
    audit_df["pain_candidate_v2"]
    &
    ~audit_df["flag_question_like_strict"]
)


print("Filtro V2 costruito.")

Filtro V2 costruito.


 15.12 — Confronto filtro v1 vs v2

In [15]:
print(
    "Candidate V1:",
    int(audit_df["pain_candidate"].sum())
)

print(
    "Candidate V2 prima QC:",
    int(audit_df["pain_candidate_v2"].sum())
)

print(
    "Candidate V2 dopo esclusione domande:",
    int(audit_df["pain_candidate_v2_clean"].sum())
)


print(
    "\nNuovi turni recuperati:",
    int(
        (
            audit_df["pain_candidate_v2_clean"]
            &
            ~audit_df["pain_candidate"]
        ).sum()
    )
)

patients_v2 = set(
    audit_df.loc[
        audit_df["pain_candidate_v2_clean"],
        "patient_id"
    ]
)

patients_without_v2 = sorted(
    set(audit_df["patient_id"])
    -
    patients_v2
)

print(
    "\nPazienti con almeno un candidate V2:",
    len(patients_v2)
)

print(
    "Pazienti ancora senza candidate:",
    len(patients_without_v2)
)

print(
    "ID ancora senza candidate:",
    patients_without_v2
)

Candidate V1: 299
Candidate V2 prima QC: 314
Candidate V2 dopo esclusione domande: 310

Nuovi turni recuperati: 15

Pazienti con almeno un candidate V2: 81
Pazienti ancora senza candidate: 9
ID ancora senza candidate: [16, 18, 26, 36, 47, 50, 56, 93, 116]


15.13 — Audit dei nuovi candidate introdotti dalla V2

In [16]:
new_v2_candidates = (
    audit_df[
        audit_df["pain_candidate_v2_clean"]
        &
        ~audit_df["pain_candidate"]
    ]
    [
        [
            "patient_id",
            "recording_id",
            "turn_id",
            "transcript_word_count",
            "pain_strong_descriptor",
            "pain_body_term",
            "pain_context_term",
            "pain_body_context",
            "transcript"
        ]
    ]
    .copy()
)

print(
    "Nuovi candidate V2:",
    len(new_v2_candidates)
)

print(
    "Pazienti coinvolti:",
    new_v2_candidates["patient_id"].nunique()
)

pd.set_option(
    "display.max_colwidth",
    500
)

display(
    new_v2_candidates
    .sample(
        n=min(50, len(new_v2_candidates)),
        random_state=42
    )
    .sort_values(
        ["patient_id", "turn_id"]
    )
    .reset_index(drop=True)
)

Nuovi candidate V2: 15
Pazienti coinvolti: 14


,patient_id,recording_id,turn_id,transcript_word_count,pain_strong_descriptor,pain_body_term,pain_context_term,pain_body_context,transcript
0,27,ID27.SStudioRuggi,ID27.SStudioRuggi__patient_turn_0038,10,False,True,True,True,si discaldava la schiena e quindi cominciava a stare meglio.
1,38,ID38.StudioRuggi,ID38.StudioRuggi__patient_turn_0033,3,True,False,False,False,incontri nel martello
2,48,ID48.StudioRuggi-001,ID48.StudioRuggi-001__patient_turn_0025,12,False,True,True,True,erano i sintomi della chemioterapia che peggioravano il fisico ma anche l'umore.
3,67,ID67.StudioRuggi,ID67.StudioRuggi__patient_turn_0028,42,False,True,True,True,"La condizione che negli ultimi mesi si verifica quasi giornalmente. Quando mi alzo ho un po' di difficoltà, diciamo in articolazioni, più calto quelle superiore, non tanto quelle inferiori, cioè le gambe non mi tanno problemi né alle ginocchie né alle alche."
4,71,ID71.StudioRuggi,ID71.StudioRuggi__patient_turn_0053,11,True,False,False,False,è solo questo formiglio che mi davanti e niente da lì
5,72,ID72.StudioRuggi,ID72.StudioRuggi__patient_turn_0028,32,False,True,True,True,"è forte, sul ginocchio è molto forte, proprio, non riesco a dormire, è un massimo onor, un'ora e mezzo, poi mi sveglio, devo stare per, per lo meno tre quarti d'or, un'ora."
6,90,ID90.StudioRuggi,ID90.StudioRuggi__patient_turn_0009,16,False,True,True,True,"Non saprei descriverlo, parte dall'articolazione tempo che l'ho mandito, l'ha mandito e la volta alla maschera."
7,103,ID103.StudioRuggi,ID103.StudioRuggi__patient_turn_0035,39,False,True,True,True,"Sono tanti e non sono semplici da descrivere. Sicuramente ciò che mi caratterizza è il fatto di essere temporale, si alternano. E non sempre lo localizzate allo stesso punto. Cioè come se fosse ritenerante. Lungo l'arto, lungo la gamba."
8,117,ID117.StudioRuggi,ID117.StudioRuggi__patient_turn_0029,8,False,True,True,True,è calore e fortissimo alla schiena e tipo...
9,130,ID130.StudioRuggi,ID130.StudioRuggi__patient_turn_0016,28,False,True,True,True,"e la cosa più imbalidante per me è la difficoltà nella guida perché ho problemi e sensibilità al piede del resto, soprattutto quando guido quindi pedale, ecceveratore, freno"


15.14 — Filtro pain-related V3
- Versione conservativa / alta precisione

In [17]:
# BASE:
# filtro V1, ma tolgo:
# - domande chiaramente attribuibili al medico o voce registrata
# - dolore esplicitamente emotivo/psicologico

audit_df["pain_core_v3"] = (
    audit_df["pain_candidate"]
    &
    ~audit_df["flag_question_like_strict"]
    &
    ~audit_df["flag_emotional_pain"]
)

# ESTENSIONI AD ALTA CONFIDENZA
# 1. Errori ASR compatibili con "formicolio"
#    osservati realmente nel dataset: es. "formiglio"
PATTERN_FORMICOLIO_ASR = re.compile(
    r"\bformigl\w*\b",
    flags=re.IGNORECASE
)

audit_df["pain_formicolio_asr"] = (
    audit_df["transcript_norm"]
    .str.contains(
        PATTERN_FORMICOLIO_ASR,
        regex=True,
        na=False
    )
)

# 2. Metafora molto specifica osservata: "chiodo ... martello"
# NON considero "martello" da solo.
PATTERN_PAIN_METAPHOR = re.compile(
    r"\bchiodo\b.{0,50}\bmartello\b"
    r"|\bmartello\b.{0,50}\bchiodo\b",
    flags=re.IGNORECASE
)

audit_df["pain_metaphor"] = (
    audit_df["transcript_norm"]
    .str.contains(
        PATTERN_PAIN_METAPHOR,
        regex=True,
        na=False
    )
)

# 3. Intensità / descrizione + localizzazione corporea
# Qui servono DUE segnali:
# una parte corporea + un'espressione fortemente compatibile con la descrizione del dolore.
PATTERN_HIGH_CONF_CONTEXT = re.compile(
    r"\bfortissim\w*\b"
    r"|\bmolto\s+forte\b"
    r"|\btroppo\s+forte\b"
    r"|\bnon\s+riesco\s+(?:a\s+)?dorm\w*\b"
    r"|\bnon\s+(?:so|saprei|riesco)\s+(?:a\s+)?descriver\w*\b"
    r"|\bcalore\b",
    flags=re.IGNORECASE
)

audit_df["pain_high_conf_context"] = (
    audit_df["transcript_norm"]
    .str.contains(
        PATTERN_HIGH_CONF_CONTEXT,
        regex=True,
        na=False
    )
)

audit_df["pain_body_high_conf"] = (
    audit_df["pain_body_term"]
    &
    audit_df["pain_high_conf_context"]
)

# PAIN-RELATED V3
audit_df["pain_related_v3"] = (
    audit_df["pain_core_v3"]
    |
    audit_df["pain_formicolio_asr"]
    |
    audit_df["pain_metaphor"]
    |
    audit_df["pain_body_high_conf"]
)

print("Filtro V3 costruito.")

Filtro V3 costruito.


15.15 — Confronto dei filtri



In [18]:
print(
    "V1 originale:",
    int(audit_df["pain_candidate"].sum())
)

print(
    "V2 clean:",
    int(audit_df["pain_candidate_v2_clean"].sum())
)

print(
    "V3 alta precisione:",
    int(audit_df["pain_related_v3"].sum())
)


patients_v3 = set(
    audit_df.loc[
        audit_df["pain_related_v3"],
        "patient_id"
    ]
)

patients_without_v3 = sorted(
    set(audit_df["patient_id"])
    -
    patients_v3
)

print(
    "\nPazienti con almeno un pain-related V3:",
    len(patients_v3)
)

print(
    "Pazienti senza pain-related V3:",
    len(patients_without_v3)
)

print(
    "ID senza pain-related:",
    patients_without_v3
)

# Turni aggiunti dalla V3 rispetto al core V1 pulito
new_v3 = (
    audit_df["pain_related_v3"]
    &
    ~audit_df["pain_core_v3"]
)

print(
    "\nNuovi turni aggiunti rispetto al core V1:",
    int(new_v3.sum())
)

display(
    audit_df.loc[
        new_v3,
        [
            "patient_id",
            "turn_id",
            "pain_formicolio_asr",
            "pain_metaphor",
            "pain_body_high_conf",
            "transcript"
        ]
    ]
    .sort_values(
        ["patient_id", "turn_id"]
    )
    .reset_index(drop=True)
)

V1 originale: 299
V2 clean: 310
V3 alta precisione: 301

Pazienti con almeno un pain-related V3: 80
Pazienti senza pain-related V3: 10
ID senza pain-related: [16, 18, 26, 27, 36, 47, 50, 56, 93, 116]

Nuovi turni aggiunti rispetto al core V1: 7


,patient_id,turn_id,pain_formicolio_asr,pain_metaphor,pain_body_high_conf,transcript
0,51,ID51.StudioRuggi__patient_turn_0021,False,False,True,"non è che oltre avevo un rosso del oro dei fastili più caldo, i fastili allegandemi principalmente, una soprazione eccessiva, ogni tanto nelle gambate di calore."
1,71,ID71.StudioRuggi__patient_turn_0053,True,False,False,è solo questo formiglio che mi davanti e niente da lì
2,72,ID72.StudioRuggi__patient_turn_0028,False,False,True,"è forte, sul ginocchio è molto forte, proprio, non riesco a dormire, è un massimo onor, un'ora e mezzo, poi mi sveglio, devo stare per, per lo meno tre quarti d'or, un'ora."
3,90,ID90.StudioRuggi__patient_turn_0009,False,False,True,"Non saprei descriverlo, parte dall'articolazione tempo che l'ho mandito, l'ha mandito e la volta alla maschera."
4,117,ID117.StudioRuggi__patient_turn_0029,False,False,True,è calore e fortissimo alla schiena e tipo...
5,117,ID117.StudioRuggi__patient_turn_0032,False,False,True,forte calore e che ti puncia dietro la schiena che non ti fanno anche camminare ti devi riposare per continuare a camminare
6,145,ID145.StudioRuggi__patient_turn_0038,False,True,False,tipo un chiodo con un martello.


15.16 — Casi V2 esclusi dalla V3

In [19]:
v2_only = (
    audit_df["pain_candidate_v2_clean"]
    &
    ~audit_df["pain_related_v3"]
)

print(
    "Turni V2 esclusi dalla V3:",
    int(v2_only.sum())
)

display(
    audit_df.loc[
        v2_only,
        [
            "patient_id",
            "turn_id",
            "pain_strong_descriptor",
            "pain_body_context",
            "transcript"
        ]
    ]
    .sort_values(
        ["patient_id", "turn_id"]
    )
    .reset_index(drop=True)
)

Turni V2 esclusi dalla V3: 11


,patient_id,turn_id,pain_strong_descriptor,pain_body_context,transcript
0,27,ID27.SStudioRuggi__patient_turn_0038,False,True,si discaldava la schiena e quindi cominciava a stare meglio.
1,38,ID38.StudioRuggi__patient_turn_0033,True,False,incontri nel martello
2,48,ID48.StudioRuggi-001__patient_turn_0025,False,True,erano i sintomi della chemioterapia che peggioravano il fisico ma anche l'umore.
3,48,ID48.StudioRuggi-001__patient_turn_0034,False,False,"e quando ho saputo di dover fare le chemioterapie ho provato un dolore interiore più che fisico, è emotivo, fortissimo."
4,67,ID67.StudioRuggi__patient_turn_0028,False,True,"La condizione che negli ultimi mesi si verifica quasi giornalmente. Quando mi alzo ho un po' di difficoltà, diciamo in articolazioni, più calto quelle superiore, non tanto quelle inferiori, cioè le gambe non mi tanno problemi né alle ginocchie né alle alche."
5,103,ID103.StudioRuggi__patient_turn_0035,False,True,"Sono tanti e non sono semplici da descrivere. Sicuramente ciò che mi caratterizza è il fatto di essere temporale, si alternano. E non sempre lo localizzate allo stesso punto. Cioè come se fosse ritenerante. Lungo l'arto, lungo la gamba."
6,130,ID130.StudioRuggi__patient_turn_0016,False,True,"e la cosa più imbalidante per me è la difficoltà nella guida perché ho problemi e sensibilità al piede del resto, soprattutto quando guido quindi pedale, ecceveratore, freno"
7,154,ID154.StudioRuggi__patient_turn_0034,True,False,martello
8,156,ID156.StudioRuggi__patient_turn_0028,False,True,"mi fa girare con difficoltà, mi fa diventare prudente nel fare delle azioni come a passare il capo o chinare il capo o girarmi a testa a sinistra, lo affronto sempre con una certa paura perché temo di"
9,156,ID156.StudioRuggi__patient_turn_0039,True,False,"Per le 8 circa il 2010, 2012 che sono andata proprio in incardinazione e quindi mi è rimasta la mandibola bloccata poi con una manovra fatta da un medico del pronto soccurso che è un oxidio facciato"


15.17 — Distribuzione definitiva dei turni pain-related

In [20]:
pain_df = (
    audit_df[
        audit_df["pain_related_v3"]
    ]
    .copy()
)

print(
    "Turni pain-related definitivi:",
    len(pain_df)
)

print(
    "Pazienti rappresentati:",
    pain_df["patient_id"].nunique()
)

print(
    "Percentuale sui turni trascritti:",
    round(
        100
        * len(pain_df)
        / int(audit_df["has_transcript"].sum()),
        2
    ),
    "%"
)

# Numero di turni pain-related per paziente
pain_counts = (
    pain_df
    .groupby("patient_id")
    .size()
    .rename("n_pain_turns")
    .reset_index()
)

print(
    "\nDistribuzione numero di pain-turn per paziente:"
)

display(
    pain_counts[
        "n_pain_turns"
    ]
    .describe()
    .to_frame()
)

# Fasce
pain_counts["pain_turn_class"] = pd.cut(
    pain_counts["n_pain_turns"],
    bins=[
        0,
        1,
        2,
        3,
        5,
        10,
        np.inf
    ],
    labels=[
        "1",
        "2",
        "3",
        "4-5",
        "6-10",
        ">10"
    ]
)

print(
    "\nPazienti per numero di pain-turn:"
)

display(
    pain_counts[
        "pain_turn_class"
    ]
    .value_counts(sort=False)
    .rename("n_patients")
    .to_frame()
)

Turni pain-related definitivi: 301
Pazienti rappresentati: 80
Percentuale sui turni trascritti: 8.11 %

Distribuzione numero di pain-turn per paziente:


,n_pain_turns
count,80.000000
mean,3.762500
std,2.571724
min,1.000000
25%,2.000000
50%,3.000000
75%,5.000000
max,11.000000



Pazienti per numero di pain-turn:


,n_patients
pain_turn_class,
1,19
2,14
3,9
4-5,19
6-10,18
>10,1


15.18 — Quota pain-related per paziente

In [21]:
all_turn_counts = (
    audit_df
    .groupby("patient_id")
    .size()
    .rename("n_total_turns")
    .reset_index()
)

patient_pain_summary = (
    all_turn_counts
    .merge(
        pain_counts[
            [
                "patient_id",
                "n_pain_turns"
            ]
        ],
        on="patient_id",
        how="left"
    )
)

patient_pain_summary["n_pain_turns"] = (
    patient_pain_summary[
        "n_pain_turns"
    ]
    .fillna(0)
    .astype(int)
)

patient_pain_summary["pain_turn_share"] = (
    patient_pain_summary["n_pain_turns"]
    /
    patient_pain_summary["n_total_turns"]
)

print(
    "Riepilogo quota pain-related per paziente:"
)

display(
    patient_pain_summary[
        [
            "n_total_turns",
            "n_pain_turns",
            "pain_turn_share"
        ]
    ]
    .describe()
)

print(
    "\nPazienti con più pain-turn:"
)

display(
    patient_pain_summary
    .sort_values(
        "n_pain_turns",
        ascending=False
    )
    .head(15)
)

Riepilogo quota pain-related per paziente:


,n_total_turns,n_pain_turns,pain_turn_share
count,90.000000,90.000000,90.000000
mean,42.500000,3.344444,0.075794
std,17.690377,2.698985,0.050328
min,8.000000,0.000000,0.000000
25%,31.250000,1.000000,0.033621
50%,39.000000,3.000000,0.073171
75%,51.750000,5.000000,0.111111
max,98.000000,11.000000,0.189189



Pazienti con più pain-turn:


,patient_id,n_total_turns,n_pain_turns,pain_turn_share
60,87,64,11,0.171875
28,42,93,10,0.107527
84,149,64,10,0.156250
76,122,71,9,0.126761
80,131,98,9,0.091837
46,62,52,8,0.153846
21,30,87,8,0.091954
53,72,55,8,0.145455
49,67,55,7,0.127273
45,61,39,7,0.179487


15.19 — Feature comportamentali disponibili nei pain-turn

In [22]:
BEHAVIOR_FEATURES = [
    "speech_activity_ratio",
    "internal_pause_mean_seconds",
    "words_per_second",
    "filler_rate_per_100_words",
    "repetition_rate_per_100_words",
    "f0_std",
    "f0_iqr"
]

print(
    "Feature comportamentali:",
    BEHAVIOR_FEATURES
)

missing_behavior = (
    pain_df[
        BEHAVIOR_FEATURES
    ]
    .isna()
    .mean()
    .mul(100)
    .sort_values(
        ascending=False
    )
    .rename("missing_percent")
    .to_frame()
)

display(
    missing_behavior.round(2)
)

Feature comportamentali: ['speech_activity_ratio', 'internal_pause_mean_seconds', 'words_per_second', 'filler_rate_per_100_words', 'repetition_rate_per_100_words', 'f0_std', 'f0_iqr']


,missing_percent
f0_iqr,1.0
f0_std,1.0
speech_activity_ratio,0.0
words_per_second,0.0
internal_pause_mean_seconds,0.0
repetition_rate_per_100_words,0.0
filler_rate_per_100_words,0.0


15.20 — Salvataggio filtro pain-related definitivo

In [23]:
PAIN_DIR = (
    RESULTS_DIR
    / "analisi_pain_related"
)

PAIN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Salvataggio dei 301 turni selezionati
pain_df.to_csv(
    PAIN_DIR
    / "turni_pain_related_v3_definitivi.csv",
    index=False
)

# Salvataggio riepilogo patient-level
patient_pain_summary.to_csv(
    PAIN_DIR
    / "riepilogo_pain_related_patient_level.csv",
    index=False
)

print(
    "Cartella risultati:",
    PAIN_DIR
)

print(
    "\nTurni pain-related salvati:",
    len(pain_df)
)

print(
    "Pazienti:",
    pain_df["patient_id"].nunique()
)

Cartella risultati: C:\Users\acer\Desktop\ProgettoTesi\risultati\analisi_pain_related

Turni pain-related salvati: 301
Pazienti: 80


15.21 — Bilanciamento del contributo dei pazienti

In [24]:
MAX_PAIN_TURNS_PER_PATIENT = 5
RANDOM_STATE = 42

rng = np.random.default_rng(
    RANDOM_STATE
)

pain_fit_df = pain_df.copy()

# Numero casuale riproducibile per ogni turno
pain_fit_df["_random_order"] = (
    rng.random(
        len(pain_fit_df)
    )
)

# Ordine casuale all'interno del paziente
pain_fit_df = (
    pain_fit_df
    .sort_values(
        [
            "patient_id",
            "_random_order"
        ]
    )
    .groupby(
        "patient_id",
        group_keys=False
    )
    .head(
        MAX_PAIN_TURNS_PER_PATIENT
    )
    .drop(
        columns="_random_order"
    )
    .reset_index(drop=True)
)

# Controlli
fit_counts = (
    pain_fit_df
    .groupby("patient_id")
    .size()
    .rename("n_turns_fit")
)

print(
    "Turni pain-related originali:",
    len(pain_df)
)

print(
    "Turni usati per fit clustering:",
    len(pain_fit_df)
)

print(
    "Turni esclusi dal bilanciamento:",
    len(pain_df) - len(pain_fit_df)
)

print(
    "Percentuale mantenuta:",
    round(
        100
        * len(pain_fit_df)
        / len(pain_df),
        2
    ),
    "%"
)

print(
    "\nPazienti nel campione:",
    pain_fit_df["patient_id"].nunique()
)

print(
    "Massimo turni per paziente:",
    fit_counts.max()
)

print(
    "Minimo turni per paziente:",
    fit_counts.min()
)

print(
    "\nDistribuzione dopo bilanciamento:"
)

display(
    fit_counts
    .describe()
    .to_frame()
)

# Quanti pazienti sono stati effettivamente limitati?
n_capped = int(
    (
        pain_counts["n_pain_turns"]
        >
        MAX_PAIN_TURNS_PER_PATIENT
    )
    .sum()
)

print(
    "\nPazienti con >5 pain-turn prima del bilanciamento:",
    n_capped
)

# Verifiche di sicurezza
assert (
    pain_fit_df["patient_id"].nunique()
    ==
    pain_df["patient_id"].nunique()
)
assert (
    fit_counts.max()
    <=
    MAX_PAIN_TURNS_PER_PATIENT
)

print(
    "\nControlli superati."
)

Turni pain-related originali: 301
Turni usati per fit clustering: 253
Turni esclusi dal bilanciamento: 48
Percentuale mantenuta: 84.05 %

Pazienti nel campione: 80
Massimo turni per paziente: 5
Minimo turni per paziente: 1

Distribuzione dopo bilanciamento:


,n_turns_fit
count,80.00000
mean,3.16250
std,1.61828
min,1.00000
25%,2.00000
50%,3.00000
75%,5.00000
max,5.00000



Pazienti con >5 pain-turn prima del bilanciamento: 19

Controlli superati.


15.22 — Preprocessing feature comportamentali

In [25]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

BEHAVIOR_FEATURES = [
    "speech_activity_ratio",
    "internal_pause_mean_seconds",
    "words_per_second",
    "filler_rate_per_100_words",
    "repetition_rate_per_100_words",
    "f0_std",
    "f0_iqr"
]

LOG_FEATURES = [
    "internal_pause_mean_seconds",
    "filler_rate_per_100_words",
    "repetition_rate_per_100_words"
]

# Partiamo dal campione bilanciato dei 253 turni
X_fit_raw = (
    pain_fit_df[
        BEHAVIOR_FEATURES
    ]
    .copy()
)

# 1. Winsorization
# Limiti calcolati SOLO sul campione usato per il fit.
# Mantengo i valori tra 1° e 99° percentile.

winsor_limits = {}

X_fit_winsor = X_fit_raw.copy()

for col in BEHAVIOR_FEATURES:
    q_low = X_fit_raw[col].quantile(0.01)
    q_high = X_fit_raw[col].quantile(0.99)
    winsor_limits[col] = (
        q_low,
        q_high
    )
    X_fit_winsor[col] = (
        X_fit_winsor[col]
        .clip(
            lower=q_low,
            upper=q_high
        )
    )

# 2. Log1p per feature fortemente asimmetriche
X_fit_transformed = (
    X_fit_winsor.copy()
)

for col in LOG_FEATURES:
    X_fit_transformed[col] = (
        np.log1p(
            X_fit_transformed[col]
            .clip(lower=0)
        )
    )

# 3. Imputazione mediana
behavior_imputer = SimpleImputer(
    strategy="median"
)


X_fit_imputed = (
    behavior_imputer
    .fit_transform(
        X_fit_transformed
    )
)

# 4. RobustScaler
behavior_scaler = RobustScaler()

X_fit_scaled = (
    behavior_scaler
    .fit_transform(
        X_fit_imputed
    )
)

print(
    "Shape matrice clustering:",
    X_fit_scaled.shape
)

print(
    "NaN finali:",
    int(
        np.isnan(
            X_fit_scaled
        ).sum()
    )
)

print(
    "Inf finali:",
    int(
        np.isinf(
            X_fit_scaled
        ).sum()
    )
)

print(
    "\nFeature utilizzate:"
)

for feature in BEHAVIOR_FEATURES:
    print(
        "-",
        feature
    )

Shape matrice clustering: (253, 7)
NaN finali: 0
Inf finali: 0

Feature utilizzate:
- speech_activity_ratio
- internal_pause_mean_seconds
- words_per_second
- filler_rate_per_100_words
- repetition_rate_per_100_words
- f0_std
- f0_iqr


15.23 — Valutazione numero di cluster

In [26]:
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

cluster_evaluation = []

for k in range(2, 7):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50
    )
    labels = model.fit_predict(
        X_fit_scaled
    )
    silhouette = silhouette_score(
        X_fit_scaled,
        labels
    )
    calinski = calinski_harabasz_score(
        X_fit_scaled,
        labels
    )
    davies = davies_bouldin_score(
        X_fit_scaled,
        labels
    )
    counts = pd.Series(
        labels
    ).value_counts()

    cluster_evaluation.append(
        {
            "k": k,
            "silhouette": silhouette,
            "calinski_harabasz": calinski,
            "davies_bouldin": davies,
            "smallest_cluster_n": int(
                counts.min()
            ),
            "largest_cluster_n": int(
                counts.max()
            )
        }
    )

cluster_evaluation_df = pd.DataFrame(
    cluster_evaluation
)

print(
    "Valutazione KMeans sui pain-related turn:"
)

display(
    cluster_evaluation_df
    .round(4)
)

Valutazione KMeans sui pain-related turn:


,k,silhouette,calinski_harabasz,davies_bouldin,smallest_cluster_n,largest_cluster_n
0,2,0.2849,87.3239,1.5206,72,181
1,3,0.2504,89.1676,1.3285,53,110
2,4,0.2326,85.1133,1.2639,31,83
3,5,0.2197,77.6043,1.2925,28,76
4,6,0.2327,75.7202,1.2904,26,53


15.24 — Stabilità K=2 vs K=3 su campionamenti diversi

In [27]:
from sklearn.metrics import adjusted_rand_score

STABILITY_SEEDS = [
    42, 52, 62, 72, 82,
    92, 102, 112, 122, 132,
    142, 152, 162, 172, 182,
    192, 202, 212, 222, 232
]

K_CANDIDATES = [2, 3]

def build_balanced_pain_sample(
    df,
    max_turns=5,
    seed=42
):

    rng = np.random.default_rng(seed)
    temp = df.copy()
    temp["_random_order"] = rng.random(
        len(temp)
    )
    temp = (
        temp
        .sort_values(
            [
                "patient_id",
                "_random_order"
            ]
        )
        .groupby(
            "patient_id",
            group_keys=False
        )
        .head(max_turns)
        .drop(
            columns="_random_order"
        )
        .reset_index(drop=True)
    )

    return temp

def fit_behavior_preprocessing(
    fit_df
):
    X_raw = (
        fit_df[
            BEHAVIOR_FEATURES
        ]
        .copy()
    )
    limits = {}
    X_winsor = X_raw.copy()

    for col in BEHAVIOR_FEATURES:
        q_low = X_raw[col].quantile(0.01)
        q_high = X_raw[col].quantile(0.99)
        limits[col] = (
            q_low,
            q_high
        )
        X_winsor[col] = (
            X_winsor[col]
            .clip(
                lower=q_low,
                upper=q_high
            )
        )
    X_transformed = X_winsor.copy()

    for col in LOG_FEATURES:
        X_transformed[col] = (
            np.log1p(
                X_transformed[col]
                .clip(lower=0)
            )
        )

    imputer = SimpleImputer(
        strategy="median"
    )

    X_imputed = (
        imputer
        .fit_transform(
            X_transformed
        )
    )

    scaler = RobustScaler()

    X_scaled = (
        scaler
        .fit_transform(
            X_imputed
        )
    )

    return (
        X_scaled,
        limits,
        imputer,
        scaler
    )

def transform_behavior_data(
    df,
    limits,
    imputer,
    scaler
):

    X = (
        df[
            BEHAVIOR_FEATURES
        ]
        .copy()
    )

    for col in BEHAVIOR_FEATURES:
        q_low, q_high = limits[col]
        X[col] = (
            X[col]
            .clip(
                lower=q_low,
                upper=q_high
            )
        )

    for col in LOG_FEATURES:
        X[col] = (
            np.log1p(
                X[col]
                .clip(lower=0)
            )
        )

    X_imp = imputer.transform(X)
    X_scaled = scaler.transform(
        X_imp
    )
    return X_scaled

# RIPETIZIONI
stability_results = []
all_predictions = {}

for k in K_CANDIDATES:
    print(
        "\n",
        "=" * 70
    )
    print(
        f"STABILITÀ K={k}"
    )
    print(
        "=" * 70
    )
    for seed in STABILITY_SEEDS:

        fit_seed_df = (
            build_balanced_pain_sample(
                pain_df,
                max_turns=5,
                seed=seed
            )
        )

        (
            X_seed_fit,
            seed_limits,
            seed_imputer,
            seed_scaler
        ) = fit_behavior_preprocessing(
            fit_seed_df
        )

        model = KMeans(
            n_clusters=k,
            random_state=seed,
            n_init=50
        )

        fit_labels = (
            model.fit_predict(
                X_seed_fit
            )
        )

        silhouette = (
            silhouette_score(
                X_seed_fit,
                fit_labels
            )
        )

        # Assegniamo i cluster a TUTTI i 301 pain-turn
        X_all_seed = (
            transform_behavior_data(
                pain_df,
                seed_limits,
                seed_imputer,
                seed_scaler
            )
        )

        labels_all = (
            model.predict(
                X_all_seed
            )
        )

        all_predictions[
            (k, seed)
        ] = labels_all

        counts = (
            pd.Series(
                labels_all
            )
            .value_counts()
        )

        stability_results.append(
            {
                "k": k,
                "seed": seed,
                "silhouette_fit": silhouette,
                "smallest_cluster_all": int(
                    counts.min()
                ),
                "largest_cluster_all": int(
                    counts.max()
                )
            }
        )

stability_results_df = (
    pd.DataFrame(
        stability_results
    )
)

# ARI rispetto alla soluzione seed 42
ari_rows = []

for k in K_CANDIDATES:
    reference = (
        all_predictions[
            (k, 42)
        ]
    )

    for seed in STABILITY_SEEDS:
        current = (
            all_predictions[
                (k, seed)
            ]
        )
        ari = adjusted_rand_score(
            reference,
            current
        )
        ari_rows.append(
            {
                "k": k,
                "seed": seed,
                "ARI_vs_seed42": ari
            }
        )

ari_df = pd.DataFrame(
    ari_rows
)

# RIEPILOGO
stability_summary = (
    stability_results_df
    .groupby("k")
    .agg(
        silhouette_mean=(
            "silhouette_fit",
            "mean"
        ),
        silhouette_std=(
            "silhouette_fit",
            "std"
        ),
        smallest_cluster_mean=(
            "smallest_cluster_all",
            "mean"
        )
    )
    .reset_index()
)

ari_summary = (
    ari_df
    .groupby("k")[
        "ARI_vs_seed42"
    ]
    .agg(
        [
            "mean",
            "std",
            "min",
            "max"
        ]
    )
    .reset_index()
)

print(
    "\nSTABILITÀ DELLA STRUTTURA:"
)

display(
    stability_summary.round(4)
)

print(
    "\nADJUSTED RAND INDEX rispetto al seed 42:"
)

display(
    ari_summary.round(4)
)


STABILITÀ K=2

STABILITÀ K=3

STABILITÀ DELLA STRUTTURA:


,k,silhouette_mean,silhouette_std,smallest_cluster_mean
0,2,0.2719,0.0181,93.65
1,3,0.2532,0.0075,66.35



ADJUSTED RAND INDEX rispetto al seed 42:


,k,mean,std,min,max
0,2,0.8816,0.1237,0.5852,1.0
1,3,0.9459,0.0630,0.8060,1.0


 15.25 — Clustering definitivo dei pain-related turn
         K = 3

In [28]:
FINAL_K = 3
FINAL_RANDOM_STATE = 42


pain_kmeans = KMeans(
    n_clusters=FINAL_K,
    random_state=FINAL_RANDOM_STATE,
    n_init=50
)


# ------------------------------------------------------------
# Fit sui 253 turni bilanciati
# ------------------------------------------------------------

pain_fit_labels = (
    pain_kmeans.fit_predict(
        X_fit_scaled
    )
)


pain_fit_df = pain_fit_df.copy()

pain_fit_df[
    "pain_behavior_cluster"
] = pain_fit_labels


print(
    "Distribuzione cluster nel campione di fit:"
)

display(
    pain_fit_df[
        "pain_behavior_cluster"
    ]
    .value_counts()
    .sort_index()
    .rename("n_turns")
    .to_frame()
)


print(
    "\nSilhouette finale:",
    round(
        silhouette_score(
            X_fit_scaled,
            pain_fit_labels
        ),
        4
    )
)

Distribuzione cluster nel campione di fit:


,n_turns
pain_behavior_cluster,
0,53
1,90
2,110



Silhouette finale: 0.2504


15.26 — Assegnazione cluster a tutti i pain-turn

In [29]:
X_all_pain = (
    pain_df[
        BEHAVIOR_FEATURES
    ]
    .copy()
)

# Winsorization con limiti appresi sul fit
for col in BEHAVIOR_FEATURES:
    q_low, q_high = (
        winsor_limits[col]
    )
    X_all_pain[col] = (
        X_all_pain[col]
        .clip(
            lower=q_low,
            upper=q_high
        )
    )

# Trasformazioni log
for col in LOG_FEATURES:

    X_all_pain[col] = (
        np.log1p(
            X_all_pain[col]
            .clip(lower=0)
        )
    )

# Imputer + scaler già addestrati
X_all_pain_imputed = (
    behavior_imputer
    .transform(
        X_all_pain
    )
)

X_all_pain_scaled = (
    behavior_scaler
    .transform(
        X_all_pain_imputed
    )
)

# Predict cluster
pain_df = pain_df.copy()
pain_df[
    "pain_behavior_cluster"
] = (
    pain_kmeans.predict(
        X_all_pain_scaled
    )
)

print(
    "Turni assegnati:",
    len(pain_df)
)

print(
    "Pazienti:",
    pain_df[
        "patient_id"
    ].nunique()
)

print(
    "\nDistribuzione cluster sui 301 turni:"
)

display(
    pain_df[
        "pain_behavior_cluster"
    ]
    .value_counts()
    .sort_index()
    .rename("n_turns")
    .to_frame()
)

Turni assegnati: 301
Pazienti: 80

Distribuzione cluster sui 301 turni:


,n_turns
pain_behavior_cluster,
0,65
1,109
2,127


15.27 — Profilo standardizzato dei cluster

In [30]:
scaled_profile_df = pd.DataFrame(
    X_all_pain_scaled,
    columns=BEHAVIOR_FEATURES,
    index=pain_df.index
)

scaled_profile_df[
    "pain_behavior_cluster"
] = (
    pain_df[
        "pain_behavior_cluster"
    ].values
)

cluster_profiles_scaled = (
    scaled_profile_df
    .groupby(
        "pain_behavior_cluster"
    )[
        BEHAVIOR_FEATURES
    ]
    .mean()
)

print(
    "Profilo standardizzato medio:"
)

display(
    cluster_profiles_scaled
    .round(3)
)

Profilo standardizzato medio:


,speech_activity_ratio,internal_pause_mean_seconds,words_per_second,filler_rate_per_100_words,repetition_rate_per_100_words,f0_std,f0_iqr
pain_behavior_cluster,,,,,,,
0,0.030,0.217,0.429,0.000,0.191,0.940,1.302
1,-0.838,1.111,-0.294,0.036,0.096,0.055,0.078
2,0.206,0.116,0.222,0.011,0.011,-0.400,-0.373


15.28 — Valori originali per cluster

In [31]:
cluster_profiles_raw = (
    pain_df
    .groupby(
        "pain_behavior_cluster"
    )[
        BEHAVIOR_FEATURES
    ]
    .median()
)

print(
    "Mediana delle feature originali:"
)

display(
    cluster_profiles_raw
    .round(4)
)

Mediana delle feature originali:


,speech_activity_ratio,internal_pause_mean_seconds,words_per_second,filler_rate_per_100_words,repetition_rate_per_100_words,f0_std,f0_iqr
pain_behavior_cluster,,,,,,,
0,0.9561,0.000,2.5765,0.0,0.0,31.1691,46.0448
1,0.8262,0.416,1.9346,0.0,0.0,21.9575,27.6245
2,0.9719,0.000,2.3197,0.0,0.0,16.4329,18.9887


15.29 — Etichette descrittive dei cluster pain-related

In [32]:
PAIN_CLUSTER_LABELS = {
    0: "alta_variabilita_prosodica",
    1: "bassa_continuita_vocale",
    2: "parlato_continuo_prosodicamente_stabile"
}

pain_df["pain_behavior_label"] = (
    pain_df["pain_behavior_cluster"]
    .map(PAIN_CLUSTER_LABELS)
)

print(
    "Distribuzione finale:"
)

display(
    pain_df[
        [
            "pain_behavior_cluster",
            "pain_behavior_label"
        ]
    ]
    .value_counts()
    .sort_index()
    .rename("n_turns")
    .to_frame()
)

Distribuzione finale:


,,n_turns
pain_behavior_cluster,pain_behavior_label,
0,alta_variabilita_prosodica,65
1,bassa_continuita_vocale,109
2,parlato_continuo_prosodicamente_stabile,127


15.30 — Copertura patient-level dei tre profili

In [34]:
cluster_patient_coverage = (
    pain_df
    .groupby(
        "pain_behavior_cluster"
    )
    .agg(
        n_turns=(
            "turn_id",
            "size"
        ),
        n_patients=(
            "patient_id",
            "nunique"
        )
    )
    .reset_index()
)

cluster_patient_coverage[
    "label"
] = (
    cluster_patient_coverage[
        "pain_behavior_cluster"
    ]
    .map(PAIN_CLUSTER_LABELS)
)

cluster_patient_coverage[
    "turn_share"
] = (
    cluster_patient_coverage["n_turns"]
    /
    len(pain_df)
)

cluster_patient_coverage[
    "patient_coverage"
] = (
    cluster_patient_coverage["n_patients"]
    /
    pain_df["patient_id"].nunique()
)

display(
    cluster_patient_coverage[
        [
            "pain_behavior_cluster",
            "label",
            "n_turns",
            "turn_share",
            "n_patients",
            "patient_coverage"
        ]
    ]
    .round(4)
)

,pain_behavior_cluster,label,n_turns,turn_share,n_patients,patient_coverage
0,0,alta_variabilita_prosodica,65,0.2159,32,0.4000
1,1,bassa_continuita_vocale,109,0.3621,41,0.5125
2,2,parlato_continuo_prosodicamente_stabile,127,0.4219,59,0.7375


15.31 — Quote dei tre profili per paziente

In [35]:
patient_cluster_counts = (
    pain_df
    .groupby(
        [
            "patient_id",
            "pain_behavior_cluster"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

# Garantiamo tutte e tre le colonne
for cluster in range(3):
    if cluster not in patient_cluster_counts.columns:
        patient_cluster_counts[cluster] = 0
patient_cluster_counts = (
    patient_cluster_counts[
        [0, 1, 2]
    ]
)

patient_cluster_counts.columns = [
    "pain_cluster_0_count",
    "pain_cluster_1_count",
    "pain_cluster_2_count"
]

patient_cluster_summary = (
    patient_cluster_counts.copy()
)

patient_cluster_summary[
    "n_pain_turns"
] = (
    patient_cluster_summary.sum(axis=1)
)

for cluster in range(3):
    patient_cluster_summary[
        f"pain_cluster_{cluster}_share"
    ] = (
        patient_cluster_summary[
            f"pain_cluster_{cluster}_count"
        ]
        /
        patient_cluster_summary[
            "n_pain_turns"
        ]
    )

patient_cluster_summary = (
    patient_cluster_summary
    .reset_index()
)

print(
    "Pazienti:",
    len(patient_cluster_summary)
)

display(
    patient_cluster_summary
    .head(15)
    .round(4)
)

Pazienti: 80


,patient_id,pain_cluster_0_count,pain_cluster_1_count,pain_cluster_2_count,n_pain_turns,pain_cluster_0_share,pain_cluster_1_share,pain_cluster_2_share
0,1,0,3,1,4,0.0000,0.75,0.2500
1,2,0,0,1,1,0.0000,0.00,1.0000
2,3,0,2,3,5,0.0000,0.40,0.6000
3,4,3,0,0,3,1.0000,0.00,0.0000
4,10,1,0,1,2,0.5000,0.00,0.5000
5,11,0,0,1,1,0.0000,0.00,1.0000
6,12,0,0,2,2,0.0000,0.00,1.0000
7,13,0,0,1,1,0.0000,0.00,1.0000
8,14,0,1,0,1,0.0000,1.00,0.0000
9,15,0,0,1,1,0.0000,0.00,1.0000


15.32 — Salvataggio definitivo clustering pain-related

In [36]:
# Turni pain-related con cluster e label
pain_df.to_csv(
    PAIN_DIR / "turni_pain_related_con_cluster_comportamentale.csv",
    index=False
)

# Profili standardizzati
cluster_profiles_scaled.to_csv(
    PAIN_DIR / "profili_cluster_pain_related_standardizzati.csv"
)

# Profili nelle unità originali
cluster_profiles_raw.to_csv(
    PAIN_DIR / "profili_cluster_pain_related_mediane_originali.csv"
)

# Copertura patient-level
cluster_patient_coverage.to_csv(
    PAIN_DIR / "copertura_cluster_pain_related_patient_level.csv",
    index=False
)

# Quote cluster per paziente
patient_cluster_summary.to_csv(
    PAIN_DIR / "profili_pain_related_patient_level.csv",
    index=False
)

# Valutazione di K
cluster_evaluation_df.to_csv(
    PAIN_DIR / "valutazione_numero_cluster_pain_related.csv",
    index=False
)

# Stabilità K=2 vs K=3
stability_results_df.to_csv(
    PAIN_DIR / "stabilita_cluster_pain_related.csv",
    index=False
)

ari_df.to_csv(
    PAIN_DIR / "stabilita_ARI_cluster_pain_related.csv",
    index=False
)

print("Risultati salvati in:")
print(PAIN_DIR)
print("\nTurni pain-related:", len(pain_df))
print("Pazienti:", pain_df["patient_id"].nunique())
print("Cluster:", pain_df["pain_behavior_cluster"].nunique())

Risultati salvati in:
C:\Users\acer\Desktop\ProgettoTesi\risultati\analisi_pain_related

Turni pain-related: 301
Pazienti: 80
Cluster: 3


15.33 — Definizione pain vs rest della conversazione

In [37]:
comparison_df = audit_df[
    audit_df["has_transcript"]
].copy()

comparison_df["speech_context"] = np.where(
    comparison_df["pain_related_v3"],
    "pain",
    "rest"
)

print(
    "Turni utilizzabili:",
    len(comparison_df)
)

print(
    "\nDistribuzione:"
)

display(
    comparison_df[
        "speech_context"
    ]
    .value_counts()
    .rename("n_turns")
    .to_frame()
)

print(
    "\nPazienti per contesto:"
)

display(
    comparison_df
    .groupby("speech_context")[
        "patient_id"
    ]
    .nunique()
    .rename("n_patients")
    .to_frame()
)

Turni utilizzabili: 3710

Distribuzione:


,n_turns
speech_context,
rest,3409
pain,301



Pazienti per contesto:


,n_patients
speech_context,
pain,80
rest,90


15.34 — Pazienti utilizzabili nel confronto within-patient

In [38]:
context_counts = (
    comparison_df
    .groupby(
        [
            "patient_id",
            "speech_context"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

for col in ["pain", "rest"]:
    if col not in context_counts.columns:
        context_counts[col] = 0

context_counts = context_counts[
    ["pain", "rest"]
]

eligible_patients = (
    context_counts[
        (context_counts["pain"] >= 1)
        &
        (context_counts["rest"] >= 1)
    ]
    .index
    .tolist()
)

print(
    "Pazienti totali:",
    comparison_df["patient_id"].nunique()
)

print(
    "Pazienti con pain-turn:",
    int((context_counts["pain"] > 0).sum())
)

print(
    "Pazienti con rest-turn:",
    int((context_counts["rest"] > 0).sum())
)

print(
    "Pazienti eleggibili per confronto paired:",
    len(eligible_patients)
)

print(
    "\nDistribuzione numero turni nei pazienti eleggibili:"
)

display(
    context_counts
    .loc[eligible_patients]
    .describe()
)

Pazienti totali: 90
Pazienti con pain-turn: 80
Pazienti con rest-turn: 90
Pazienti eleggibili per confronto paired: 80

Distribuzione numero turni nei pazienti eleggibili:


speech_context,pain,rest
count,80.000000,80.000000
mean,3.762500,38.937500
std,2.571724,16.192938
min,1.000000,10.000000
25%,2.000000,28.000000
50%,3.000000,36.500000
75%,5.000000,47.000000
max,11.000000,88.000000


15.35 — Mediane patient-level: pain vs rest

In [39]:
within_df = (
    comparison_df[
        comparison_df["patient_id"].isin(
            eligible_patients
        )
    ]
    .copy()
)

# Mediana di ogni feature per:
# paziente × contesto (pain/rest)
patient_context_medians = (
    within_df
    .groupby(
        [
            "patient_id",
            "speech_context"
        ]
    )[BEHAVIOR_FEATURES]
    .median()
    .unstack(
        "speech_context"
    )
)

# Appiattiamo le colonne
# es. f0_std_pain, f0_std_rest
patient_context_medians.columns = [
    f"{feature}_{context}"
    for feature, context
    in patient_context_medians.columns
]

patient_context_medians = (
    patient_context_medians
    .reset_index()
)

# Aggiungiamo numero di turni pain/rest
turn_counts_within = (
    context_counts
    .loc[eligible_patients]
    .reset_index()
    .rename(
        columns={
            "pain": "n_pain_turns",
            "rest": "n_rest_turns"
        }
    )
)

patient_context_medians = (
    patient_context_medians
    .merge(
        turn_counts_within,
        on="patient_id",
        how="left"
    )
)

print(
    "Pazienti totali paired:",
    len(patient_context_medians)
)

print(
    "Pazienti con almeno 2 pain-turn:",
    int(
        (
            patient_context_medians[
                "n_pain_turns"
            ] >= 2
        ).sum()
    )
)

# Completezza delle coppie
complete_rows = []

for feature in BEHAVIOR_FEATURES:
    pain_col = f"{feature}_pain"
    rest_col = f"{feature}_rest"
    complete_rows.append(
        {
            "feature": feature,
            "complete_pairs_all80": int(
                patient_context_medians[
                    [pain_col, rest_col]
                ]
                .notna()
                .all(axis=1)
                .sum()
            ),
            "complete_pairs_min2pain": int(
                (
                    (
                        patient_context_medians[
                            "n_pain_turns"
                        ] >= 2
                    )
                    &
                    patient_context_medians[
                        [pain_col, rest_col]
                    ]
                    .notna()
                    .all(axis=1)
                )
                .sum()
            )
        }
    )

complete_pairs_df = pd.DataFrame(
    complete_rows
)

display(
    complete_pairs_df
)

Pazienti totali paired: 80
Pazienti con almeno 2 pain-turn: 61


,feature,complete_pairs_all80,complete_pairs_min2pain
0,speech_activity_ratio,80,61
1,internal_pause_mean_seconds,80,61
2,words_per_second,80,61
3,filler_rate_per_100_words,80,61
4,repetition_rate_per_100_words,80,61
5,f0_std,80,61
6,f0_iqr,80,61


15.36 — Wilcoxon paired: pain vs rest

In [40]:
from scipy.stats import (
    wilcoxon,
    rankdata
)

from statsmodels.stats.multitest import (
    multipletests
)

def paired_pain_rest_analysis(
    patient_df,
    min_pain_turns,
    analysis_name
):
    subset = (
        patient_df[
            patient_df["n_pain_turns"]
            >= min_pain_turns
        ]
        .copy()
    )
    rows = []

    for feature in BEHAVIOR_FEATURES:

        pain_col = (
            f"{feature}_pain"
        )

        rest_col = (
            f"{feature}_rest"
        )

        temp = (
            subset[
                [
                    pain_col,
                    rest_col
                ]
            ]
            .dropna()
            .copy()
        )

        pain_values = (
            temp[pain_col]
            .to_numpy()
        )

        rest_values = (
            temp[rest_col]
            .to_numpy()
        )

        diff = (
            pain_values
            -
            rest_values
        )

        # Wilcoxon
        if np.allclose(
            diff,
            0
        ):

            statistic = 0.0
            p_value = 1.0
        else:
            test = wilcoxon(
                pain_values,
                rest_values,
                alternative="two-sided",
                zero_method="wilcox",
                method="auto"
            )
            statistic = (
                test.statistic
            )
            p_value = (
                test.pvalue
            )

        # Rank-biserial correlation
        # positivo -> feature maggiore nei pain-turn
        # negativo -> feature minore nei pain-turn
        nonzero_diff = (
            diff[
                diff != 0
            ]
        )


        if len(nonzero_diff) == 0:
            rank_biserial = 0.0
        else:
            ranks = rankdata(
                np.abs(
                    nonzero_diff
                )
            )
            w_positive = (
                ranks[
                    nonzero_diff > 0
                ]
                .sum()
            )
            w_negative = (
                ranks[
                    nonzero_diff < 0
                ]
                .sum()
            )
            rank_biserial = (
                (
                    w_positive
                    -
                    w_negative
                )
                /
                (
                    w_positive
                    +
                    w_negative
                )
            )

        rows.append(
            {
                "analysis": analysis_name,
                "feature": feature,
                "n_pairs": len(temp),
                "median_pain": np.median(
                    pain_values
                ),
                "median_rest": np.median(
                    rest_values
                ),
                "median_difference_pain_minus_rest":
                    np.median(diff),
                "patients_pain_higher_percent":
                    100
                    * np.mean(
                        diff > 0
                    ),
                "rank_biserial":
                    rank_biserial,
                "wilcoxon_statistic":
                    statistic,
                "p_raw":
                    p_value
            }
        )

    results = pd.DataFrame(
        rows
    )

    # Correzione FDR sulle 7 feature
    reject, q_values, _, _ = (
        multipletests(
            results["p_raw"],
            alpha=0.05,
            method="fdr_bh"
        )
    )

    results[
        "p_fdr"
    ] = q_values

    results[
        "significant_fdr_005"
    ] = reject

    return results

# ANALISI PRIMARIA
# almeno 2 pain-turn
within_primary = (
    paired_pain_rest_analysis(
        patient_context_medians,
        min_pain_turns=2,
        analysis_name="primary_min_2_pain_turns"
    )
)

print(
    "ANALISI PRIMARIA — almeno 2 pain-turn:"
)

display(
    within_primary.round(4)
)

# SENSITIVITY
# tutti gli 80 pazienti
within_sensitivity = (
    paired_pain_rest_analysis(
        patient_context_medians,
        min_pain_turns=1,
        analysis_name="sensitivity_all_pain_patients"
    )
)

print(
    "\nANALISI DI SENSIBILITÀ — tutti gli 80 pazienti:"
)

display(
    within_sensitivity.round(4)
)

ANALISI PRIMARIA — almeno 2 pain-turn:


,analysis,feature,n_pairs,median_pain,median_rest,median_difference_pain_minus_rest,patients_pain_higher_percent,rank_biserial,wilcoxon_statistic,p_raw,p_fdr,significant_fdr_005
0,primary_min_2_pain_turns,speech_activity_ratio,61,0.9285,0.9670,-0.0425,16.3934,-0.8426,144.0,0.0000,0.0000,True
1,primary_min_2_pain_turns,internal_pause_mean_seconds,61,0.0000,0.0000,0.0000,49.1803,1.0000,0.0,0.0000,0.0000,True
2,primary_min_2_pain_turns,words_per_second,61,2.1969,2.1240,0.0981,62.2951,0.2903,671.0,0.0486,0.0681,False
3,primary_min_2_pain_turns,filler_rate_per_100_words,61,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,1.0000,1.0000,False
4,primary_min_2_pain_turns,repetition_rate_per_100_words,61,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,1.0000,1.0000,False
5,primary_min_2_pain_turns,f0_std,61,20.7357,18.2362,3.2394,75.4098,0.6415,339.0,0.0000,0.0000,True
6,primary_min_2_pain_turns,f0_iqr,61,24.9239,22.8312,2.3327,65.5738,0.4585,512.0,0.0018,0.0032,True



ANALISI DI SENSIBILITÀ — tutti gli 80 pazienti:


,analysis,feature,n_pairs,median_pain,median_rest,median_difference_pain_minus_rest,patients_pain_higher_percent,rank_biserial,wilcoxon_statistic,p_raw,p_fdr,significant_fdr_005
0,sensitivity_all_pain_patients,speech_activity_ratio,80,0.9361,0.9723,-0.0418,13.75,-0.8831,171.0,0.0000,0.0000,True
1,sensitivity_all_pain_patients,internal_pause_mean_seconds,80,0.0000,0.0000,0.0000,46.25,1.0000,0.0,0.0000,0.0000,True
2,sensitivity_all_pain_patients,words_per_second,80,2.2122,2.1483,0.0970,58.75,0.2136,1274.0,0.0970,0.1358,False
3,sensitivity_all_pain_patients,filler_rate_per_100_words,80,0.0000,0.0000,0.0000,0.00,0.0000,0.0,1.0000,1.0000,False
4,sensitivity_all_pain_patients,repetition_rate_per_100_words,80,0.0000,0.0000,0.0000,0.00,0.0000,0.0,1.0000,1.0000,False
5,sensitivity_all_pain_patients,f0_std,80,20.3970,17.5395,3.1996,75.00,0.6111,630.0,0.0000,0.0000,True
6,sensitivity_all_pain_patients,f0_iqr,80,24.1009,21.9411,1.9145,65.00,0.3963,978.0,0.0021,0.0036,True


15.37 — Controllo durata e lunghezza testuale 
- pain vs rest, within-patient

In [41]:
from scipy.stats import wilcoxon

CONTROL_FEATURES = [
    "turn_duration_seconds",
    "transcript_word_count"
]

def paired_control_analysis(
    df,
    min_pain_turns,
    label
):

    patients_ok = (
        patient_context_medians.loc[
            patient_context_medians["n_pain_turns"]
            >= min_pain_turns,
            "patient_id"
        ]
        .tolist()
    )

    temp = (
        comparison_df[
            comparison_df["patient_id"]
            .isin(patients_ok)
        ]
        .copy()
    )

    rows = []

    for feature in CONTROL_FEATURES:
        medians = (
            temp
            .groupby(
                [
                    "patient_id",
                    "speech_context"
                ]
            )[feature]
            .median()
            .unstack()
            .dropna()
        )

        pain_values = (
            medians["pain"]
            .to_numpy()
        )

        rest_values = (
            medians["rest"]
            .to_numpy()
        )

        diff = (
            pain_values
            -
            rest_values
        )

        if np.allclose(diff, 0):
            stat = 0.0
            p = 1.0
        else:
            test = wilcoxon(
                pain_values,
                rest_values,
                alternative="two-sided",
                zero_method="wilcox",
                method="auto"
            )
            stat = test.statistic
            p = test.pvalue

        rows.append(
            {
                "analysis": label,
                "feature": feature,
                "n_pairs": len(medians),
                "median_pain":
                    np.median(pain_values),
                "median_rest":
                    np.median(rest_values),
                "median_difference":
                    np.median(diff),
                "pain_higher_percent":
                    100 * np.mean(diff > 0),
                "pain_lower_percent":
                    100 * np.mean(diff < 0),
                "equal_percent":
                    100 * np.mean(diff == 0),
                "wilcoxon_statistic":
                    stat,
                "p_value":
                    p
            }
        )

    return pd.DataFrame(rows)

# Analisi primaria: >= 2 pain-turn
duration_primary = (
    paired_control_analysis(
        comparison_df,
        min_pain_turns=2,
        label="primary_min_2_pain_turns"
    )
)

print(
    "CONTROLLO DURATA — ANALISI PRIMARIA:"
)

display(
    duration_primary.round(4)
)

# Sensitivity: tutti gli 80
duration_sensitivity = (
    paired_control_analysis(
        comparison_df,
        min_pain_turns=1,
        label="sensitivity_all_pain_patients"
    )
)

print(
    "\nCONTROLLO DURATA — SENSITIVITY:"
)

display(
    duration_sensitivity.round(4)
)

CONTROLLO DURATA — ANALISI PRIMARIA:


,analysis,feature,n_pairs,median_pain,median_rest,median_difference,pain_higher_percent,pain_lower_percent,equal_percent,wilcoxon_statistic,p_value
0,primary_min_2_pain_turns,turn_duration_seconds,61,4.05,1.771,2.295,95.082,4.9180,0.0000,13.0,0.0
1,primary_min_2_pain_turns,transcript_word_count,61,10.00,4.000,5.500,95.082,1.6393,3.2787,4.5,0.0



CONTROLLO DURATA — SENSITIVITY:


,analysis,feature,n_pairs,median_pain,median_rest,median_difference,pain_higher_percent,pain_lower_percent,equal_percent,wilcoxon_statistic,p_value
0,sensitivity_all_pain_patients,turn_duration_seconds,80,3.78,1.6115,2.2118,95.0,5.0,0.0,41.0,0.0
1,sensitivity_all_pain_patients,transcript_word_count,80,9.75,4.0000,5.2500,95.0,2.5,2.5,24.5,0.0


15.38 — Dettaglio direzione differenze nelle pause

In [42]:
for min_pain, name in [
    (2, "PRIMARY"),
    (1, "SENSITIVITY")
]:

    temp = (
        patient_context_medians[
            patient_context_medians[
                "n_pain_turns"
            ] >= min_pain
        ]
        .copy()
    )

    diff_pause = (
        temp[
            "internal_pause_mean_seconds_pain"
        ]
        -
        temp[
            "internal_pause_mean_seconds_rest"
        ]
    )

    print("\n", name)
    print(
        "Pause maggiori nei pain-turn:",
        int((diff_pause > 0).sum())
    )

    print(
        "Pause minori nei pain-turn:",
        int((diff_pause < 0).sum())
    )

    print(
        "Nessuna differenza:",
        int((diff_pause == 0).sum())
    )


 PRIMARY
Pause maggiori nei pain-turn: 30
Pause minori nei pain-turn: 0
Nessuna differenza: 31

 SENSITIVITY
Pause maggiori nei pain-turn: 37
Pause minori nei pain-turn: 0
Nessuna differenza: 43


15.39 — Duration matching within-patient
- Pain turn vs rest turn

In [44]:
from scipy.optimize import linear_sum_assignment

def duration_match_within_patient(
    df,
    patient_ids
):
    matched_rows = []
    patients_with_shortage = []

    for patient_id in patient_ids:
        pain = (
            df[
                (df["patient_id"] == patient_id)
                &
                (df["speech_context"] == "pain")
            ]
            .copy()
            .reset_index(drop=True)
        )

        rest = (
            df[
                (df["patient_id"] == patient_id)
                &
                (df["speech_context"] == "rest")
            ]
            .copy()
            .reset_index(drop=True)
        )

        if len(rest) < len(pain):
            patients_with_shortage.append(
                {
                    "patient_id": patient_id,
                    "n_pain": len(pain),
                    "n_rest": len(rest)
                }
            )

        # Matrice dei costi:
        # differenza assoluta di durata

        pain_duration = (
            pain["turn_duration_seconds"]
            .to_numpy()
        )

        rest_duration = (
            rest["turn_duration_seconds"]
            .to_numpy()
        )

        cost_matrix = np.abs(
            pain_duration[:, None]
            -
            rest_duration[None, :]
        )

        pain_idx, rest_idx = (
            linear_sum_assignment(
                cost_matrix
            )
        )

        # Salviamo le coppie
        for pair_number, (i, j) in enumerate(
            zip(
                pain_idx,
                rest_idx
            )
        ):
            row = {
                "patient_id": patient_id,
                "pair_number": pair_number,
                "pain_turn_id":
                    pain.loc[i, "turn_id"],
                "rest_turn_id":
                    rest.loc[j, "turn_id"],
                "pain_duration":
                    pain.loc[
                        i,
                        "turn_duration_seconds"
                    ],
                "rest_duration":
                    rest.loc[
                        j,
                        "turn_duration_seconds"
                    ]
            }

            row[
                "absolute_duration_difference"
            ] = abs(
                row["pain_duration"]
                -
                row["rest_duration"]
            )

            # Copiamo anche le 7 feature
            for feature in BEHAVIOR_FEATURES:
                row[
                    f"{feature}_pain"
                ] = pain.loc[
                    i,
                    feature
                ]
                row[
                    f"{feature}_rest"
                ] = rest.loc[
                    j,
                    feature
                ]

            matched_rows.append(
                row
            )

    return (
        pd.DataFrame(matched_rows),
        pd.DataFrame(
            patients_with_shortage
        )
    )

15.40 — Costruzione campioni duration-matched

In [45]:
primary_patient_ids = (
    patient_context_medians.loc[
        patient_context_medians[
            "n_pain_turns"
        ] >= 2,
        "patient_id"
    ]
    .tolist()
)

sensitivity_patient_ids = (
    patient_context_medians[
        "patient_id"
    ]
    .tolist()
)

matched_primary, shortage_primary = (
    duration_match_within_patient(
        comparison_df,
        primary_patient_ids
    )
)

matched_sensitivity, shortage_sensitivity = (
    duration_match_within_patient(
        comparison_df,
        sensitivity_patient_ids
    )
)

print("===== PRIMARY =====")
print(
    "Pazienti:",
    matched_primary[
        "patient_id"
    ].nunique()
)
print(
    "Coppie pain-rest:",
    len(matched_primary)
)
print(
    "Pazienti con rest insufficienti:",
    len(shortage_primary)
)
print(
    "\nDurata delle coppie:"
)

display(
    matched_primary[
        [
            "pain_duration",
            "rest_duration",
            "absolute_duration_difference"
        ]
    ]
    .describe()
    .round(4)
)

print("\n===== SENSITIVITY =====")
print(
    "Pazienti:",
    matched_sensitivity[
        "patient_id"
    ].nunique()
)
print(
    "Coppie pain-rest:",
    len(matched_sensitivity)
)
print(
    "Pazienti con rest insufficienti:",
    len(shortage_sensitivity)
)

display(
    matched_sensitivity[
        [
            "pain_duration",
            "rest_duration",
            "absolute_duration_difference"
        ]
    ]
    .describe()
    .round(4)
)

===== PRIMARY =====
Pazienti: 61
Coppie pain-rest: 282
Pazienti con rest insufficienti: 0

Durata delle coppie:


,pain_duration,rest_duration,absolute_duration_difference
count,282.0000,282.0000,282.0000
mean,6.7394,5.6564,1.2471
std,6.0698,4.8232,2.6285
min,0.6240,0.6910,0.0000
25%,2.6285,2.5395,0.0385
50%,4.5650,4.1680,0.1770
75%,8.7328,7.2688,0.9415
max,40.6350,34.9140,22.2920



===== SENSITIVITY =====
Pazienti: 80
Coppie pain-rest: 301
Pazienti con rest insufficienti: 0


,pain_duration,rest_duration,absolute_duration_difference
count,301.0000,301.0000,301.0000
mean,6.6545,5.5469,1.2637
std,6.0545,4.7357,2.6746
min,0.6240,0.6910,0.0000
25%,2.6160,2.5650,0.0499
50%,4.4210,4.0500,0.1680
75%,8.6570,7.1210,0.9280
max,40.6350,34.9140,22.2920


15.41 — Valutazione qualità del matching per durata

In [46]:
CALIPERS = [
    0.5,
    1.0,
    2.0
]

def evaluate_matching_calipers(
    matched_df,
    label
):

    rows = []
    total_pairs = len(
        matched_df
    )

    total_patients = (
        matched_df[
            "patient_id"
        ]
        .nunique()
    )

    for caliper in CALIPERS:
        temp = (
            matched_df[
                matched_df[
                    "absolute_duration_difference"
                ] <= caliper
            ]
            .copy()
        )

        patient_pair_counts = (
            temp
            .groupby(
                "patient_id"
            )
            .size()
        )

        rows.append(
            {
                "analysis": label,
                "caliper_seconds":
                    caliper,
                "n_pairs":
                    len(temp),
                "pair_retention_percent":
                    100
                    * len(temp)
                    / total_pairs,
                "n_patients":
                    temp[
                        "patient_id"
                    ]
                    .nunique(),
                "patient_retention_percent":
                    100
                    * temp[
                        "patient_id"
                    ]
                    .nunique()
                    / total_patients,
                "patients_with_at_least_2_pairs":
                    int(
                        (
                            patient_pair_counts
                            >= 2
                        )
                        .sum()
                    ),
                "median_abs_duration_diff":
                    temp[
                        "absolute_duration_difference"
                    ]
                    .median()
                    if len(temp) > 0
                    else np.nan,
                "mean_abs_duration_diff":
                    temp[
                        "absolute_duration_difference"
                    ]
                    .mean()
                    if len(temp) > 0
                    else np.nan
            }
        )

    return pd.DataFrame(
        rows
    )

caliper_primary = (
    evaluate_matching_calipers(
        matched_primary,
        "primary"
    )
)

caliper_sensitivity = (
    evaluate_matching_calipers(
        matched_sensitivity,
        "sensitivity"
    )
)

caliper_summary = pd.concat(
    [
        caliper_primary,
        caliper_sensitivity
    ],
    ignore_index=True
)

display(
    caliper_summary.round(3)
)

,analysis,caliper_seconds,n_pairs,pair_retention_percent,n_patients,patient_retention_percent,patients_with_at_least_2_pairs,median_abs_duration_diff,mean_abs_duration_diff
0,primary,0.5,180,63.830,56,91.803,48,0.059,0.110
1,primary,1.0,215,76.241,59,96.721,53,0.084,0.205
2,primary,2.0,234,82.979,60,98.361,56,0.101,0.296
3,sensitivity,0.5,195,64.784,71,88.750,48,0.067,0.110
4,sensitivity,1.0,230,76.412,74,92.500,53,0.084,0.199
5,sensitivity,2.0,250,83.056,76,95.000,56,0.101,0.292


15.42 — Differenza relativa di durata

In [47]:
for name, df in [
    ("PRIMARY", matched_primary),
    ("SENSITIVITY", matched_sensitivity)
]:

    temp = df.copy()
    temp[
        "relative_duration_difference"
    ] = (
        temp[
            "absolute_duration_difference"
        ]
        /
        temp[
            "pain_duration"
        ]
    )

    print(
        "\n",
        name
    )

    display(
        temp[
            "relative_duration_difference"
        ]
        .describe(
            percentiles=[
                0.25,
                0.50,
                0.75,
                0.90,
                0.95
            ]
        )
        .to_frame()
        .round(4)
    )

    print(
        "Coppie con differenza relativa <= 25%:",
        int(
            (
                temp[
                    "relative_duration_difference"
                ]
                <= 0.25
            )
            .sum()
        ),
        "/",
        len(temp)
    )

    print(
        "Pazienti rappresentati:",
        temp.loc[
            temp[
                "relative_duration_difference"
            ] <= 0.25,
            "patient_id"
        ]
        .nunique()
    )


 PRIMARY


,relative_duration_difference
count,282.0000
mean,0.1120
std,0.1610
min,0.0000
25%,0.0163
50%,0.0367
75%,0.1400
90%,0.3350
95%,0.4778
max,0.8225


Coppie con differenza relativa <= 25%: 239 / 282
Pazienti rappresentati: 60

 SENSITIVITY


,relative_duration_difference
count,301.0000
mean,0.1132
std,0.1629
min,0.0000
25%,0.0169
50%,0.0368
75%,0.1378
90%,0.3353
95%,0.4954
max,0.8225


Coppie con differenza relativa <= 25%: 254 / 301
Pazienti rappresentati: 75


15.43 — Duration matching definitivo
- Caliper assoluto <= 1 secondo

In [48]:
FINAL_DURATION_CALIPER = 1.0

matched_final = (
    matched_sensitivity[
        matched_sensitivity[
            "absolute_duration_difference"
        ] <= FINAL_DURATION_CALIPER
    ]
    .copy()
)

# Numero di coppie valide per paziente
matched_pair_counts = (
    matched_final
    .groupby("patient_id")
    .size()
    .rename("n_matched_pairs")
)

print(
    "Coppie matched definitive:",
    len(matched_final)
)

print(
    "Pazienti con almeno 1 coppia:",
    matched_pair_counts.size
)

print(
    "Pazienti con almeno 2 coppie:",
    int(
        (matched_pair_counts >= 2).sum()
    )
)

print(
    "\nDifferenza assoluta di durata:"
)

display(
    matched_final[
        "absolute_duration_difference"
    ]
    .describe()
    .to_frame()
    .round(4)
)

Coppie matched definitive: 230
Pazienti con almeno 1 coppia: 74
Pazienti con almeno 2 coppie: 53

Differenza assoluta di durata:


,absolute_duration_difference
count,230.0000
mean,0.1989
std,0.2455
min,0.0000
25%,0.0330
50%,0.0840
75%,0.2697
max,0.9950


15.44 — Analisi finale duration-matched patient-level

In [49]:
from scipy.stats import wilcoxon, rankdata
from statsmodels.stats.multitest import multipletests

# Mediane delle feature matched per paziente
matched_patient_rows = []

for patient_id, group in matched_final.groupby(
    "patient_id"
):
    row = {
        "patient_id": patient_id,
        "n_matched_pairs": len(group)
    }

    for feature in BEHAVIOR_FEATURES:
        row[
            f"{feature}_pain"
        ] = group[
            f"{feature}_pain"
        ].median()

        row[
            f"{feature}_rest"
        ] = group[
            f"{feature}_rest"
        ].median()

    matched_patient_rows.append(row)

matched_patient_df = pd.DataFrame(
    matched_patient_rows
)

print(
    "Pazienti matched:",
    len(matched_patient_df)
)

print(
    "Con >=2 coppie:",
    int(
        (
            matched_patient_df[
                "n_matched_pairs"
            ] >= 2
        ).sum()
    )
)

Pazienti matched: 74
Con >=2 coppie: 53


c:\Users\acer\Desktop\ProgettoTesi\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\acer\Desktop\ProgettoTesi\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [50]:
# Funzione Wilcoxon patient-level
def analyze_matched_patients(
    df,
    min_pairs,
    analysis_name
):

    subset = (
        df[
            df["n_matched_pairs"]
            >= min_pairs
        ]
        .copy()
    )

    rows = []

    for feature in BEHAVIOR_FEATURES:
        pain_values = (
            subset[
                f"{feature}_pain"
            ]
            .to_numpy()
        )
        rest_values = (
            subset[
                f"{feature}_rest"
            ]
            .to_numpy()
        )
        valid = (
            ~np.isnan(pain_values)
            &
            ~np.isnan(rest_values)
        )
        pain_values = pain_values[valid]
        rest_values = rest_values[valid]

        diff = (
            pain_values
            -
            rest_values
        )

        if np.allclose(diff, 0):
            statistic = 0.0
            p_value = 1.0
        else:
            test = wilcoxon(
                pain_values,
                rest_values,
                alternative="two-sided",
                zero_method="wilcox",
                method="auto"
            )

            statistic = test.statistic
            p_value = test.pvalue

        # Rank-biserial
        nonzero = diff[
            diff != 0
        ]

        if len(nonzero) == 0:
            rank_biserial = 0.0
        else:
            ranks = rankdata(
                np.abs(nonzero)
            )
            w_pos = ranks[
                nonzero > 0
            ].sum()
            w_neg = ranks[
                nonzero < 0
            ].sum()

            rank_biserial = (
                (w_pos - w_neg)
                /
                (w_pos + w_neg)
            )

        rows.append(
            {
                "analysis": analysis_name,
                "feature": feature,
                "n_patients": len(
                    pain_values
                ),
                "median_pain":
                    np.median(
                        pain_values
                    ),
                "median_rest":
                    np.median(
                        rest_values
                    ),
                "median_difference":
                    np.median(diff),
                "pain_higher_percent":
                    100
                    * np.mean(
                        diff > 0
                    ),
                "pain_lower_percent":
                    100
                    * np.mean(
                        diff < 0
                    ),
                "rank_biserial":
                    rank_biserial,
                "p_raw":
                    p_value,
                "wilcoxon_statistic":
                    statistic
            }
        )

    results = pd.DataFrame(rows)

    reject, p_fdr, _, _ = (
        multipletests(
            results["p_raw"],
            alpha=0.05,
            method="fdr_bh"
        )
    )

    results["p_fdr"] = p_fdr

    results[
        "significant_fdr_005"
    ] = reject

    return results

In [51]:
# PRIMARY
# almeno 2 coppie matched
matched_primary_results = (
    analyze_matched_patients(
        matched_patient_df,
        min_pairs=2,
        analysis_name=(
            "duration_matched_primary_min2pairs"
        )
    )
)

print(
    "ANALISI PRIMARIA DURATION-MATCHED:"
)

display(
    matched_primary_results
    .round(4)
)

# SENSITIVITY
# almeno 1 coppia matched
matched_sensitivity_results = (
    analyze_matched_patients(
        matched_patient_df,
        min_pairs=1,
        analysis_name=(
            "duration_matched_sensitivity_min1pair"
        )
    )
)

print(
    "\nSENSITIVITY DURATION-MATCHED:"
)

display(
    matched_sensitivity_results
    .round(4)
)

ANALISI PRIMARIA DURATION-MATCHED:


,analysis,feature,n_patients,median_pain,median_rest,median_difference,pain_higher_percent,pain_lower_percent,rank_biserial,p_raw,wilcoxon_statistic,p_fdr,significant_fdr_005
0,duration_matched_primary_min2pairs,speech_activity_ratio,53,0.9477,0.9457,0.0058,54.7170,43.3962,0.1451,0.3625,589.0,0.4229,False
1,duration_matched_primary_min2pairs,internal_pause_mean_seconds,53,0.0000,0.0000,0.0000,20.7547,30.1887,-0.2593,0.2390,140.0,0.4183,False
2,duration_matched_primary_min2pairs,words_per_second,53,2.2792,2.0509,0.2570,64.1509,35.8491,0.3389,0.0318,473.0,0.1113,False
3,duration_matched_primary_min2pairs,filler_rate_per_100_words,53,0.0000,0.0000,0.0000,1.8868,0.0000,1.0000,0.3173,0.0,0.4229,False
4,duration_matched_primary_min2pairs,repetition_rate_per_100_words,53,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0,1.0000,False
5,duration_matched_primary_min2pairs,f0_std,53,20.7357,18.0314,1.6768,64.1509,35.8491,0.3543,0.0248,462.0,0.1113,False
6,duration_matched_primary_min2pairs,f0_iqr,53,24.5279,21.7717,2.2240,60.3774,39.6226,0.3068,0.0520,496.0,0.1213,False



SENSITIVITY DURATION-MATCHED:


,analysis,feature,n_patients,median_pain,median_rest,median_difference,pain_higher_percent,pain_lower_percent,rank_biserial,p_raw,wilcoxon_statistic,p_fdr,significant_fdr_005
0,duration_matched_sensitivity_min1pair,speech_activity_ratio,74,0.9509,0.9544,0.0000,47.2973,45.9459,0.0087,0.9499,1197.0,0.9499,False
1,duration_matched_sensitivity_min1pair,internal_pause_mean_seconds,74,0.0000,0.0000,0.0000,17.5676,25.6757,-0.2727,0.1781,192.0,0.3117,False
2,duration_matched_sensitivity_min1pair,words_per_second,74,2.2782,1.9509,0.2023,62.1622,37.8378,0.2706,0.0431,1012.0,0.1684,False
3,duration_matched_sensitivity_min1pair,filler_rate_per_100_words,74,0.0000,0.0000,0.0000,1.3514,0.0000,1.0000,0.3173,0.0,0.3702,False
4,duration_matched_sensitivity_min1pair,repetition_rate_per_100_words,74,0.0000,0.0000,0.0000,1.3514,0.0000,1.0000,0.3173,0.0,0.3702,False
5,duration_matched_sensitivity_min1pair,f0_std,73,20.5802,18.0314,2.6237,63.0137,36.9863,0.2662,0.0481,991.0,0.1684,False
6,duration_matched_sensitivity_min1pair,f0_iqr,73,24.1030,22.0620,1.7487,57.5342,42.4658,0.2336,0.0828,1035.0,0.1933,False


15.45 — Salvataggio analisi within-patient definitiva

In [52]:
# Confronto originale non matched
within_primary.to_csv(
    PAIN_DIR / "within_patient_pain_vs_rest_primary.csv",
    index=False
)

within_sensitivity.to_csv(
    PAIN_DIR / "within_patient_pain_vs_rest_sensitivity.csv",
    index=False
)

# Controllo durata
duration_primary.to_csv(
    PAIN_DIR / "controllo_durata_primary.csv",
    index=False
)

duration_sensitivity.to_csv(
    PAIN_DIR / "controllo_durata_sensitivity.csv",
    index=False
)

# Coppie duration-matched definitive
matched_final.to_csv(
    PAIN_DIR / "coppie_pain_rest_duration_matched_1s.csv",
    index=False
)

# Dataset patient-level matched
matched_patient_df.to_csv(
    PAIN_DIR / "within_patient_duration_matched_patient_level.csv",
    index=False
)

# Risultati finali duration-matched
matched_primary_results.to_csv(
    PAIN_DIR / "duration_matched_primary_results.csv",
    index=False
)

matched_sensitivity_results.to_csv(
    PAIN_DIR / "duration_matched_sensitivity_results.csv",
    index=False
)

print("Risultati finali salvati in:")
print(PAIN_DIR)
print("\nCoppie matched:", len(matched_final))
print(
    "Pazienti >=1 coppia:",
    matched_final["patient_id"].nunique()
)
print(
    "Pazienti >=2 coppie:",
    int(
        (
            matched_patient_df["n_matched_pairs"] >= 2
        ).sum()
    )
)

Risultati finali salvati in:
C:\Users\acer\Desktop\ProgettoTesi\risultati\analisi_pain_related

Coppie matched: 230
Pazienti >=1 coppia: 74
Pazienti >=2 coppie: 53


15.46 — Audit finale Notebook 15

In [53]:
EXPECTED_FILES = [
    "turni_pain_related_v3_definitivi.csv",
    "riepilogo_pain_related_patient_level.csv",
    "turni_pain_related_con_cluster_comportamentale.csv",
    "profili_cluster_pain_related_standardizzati.csv",
    "profili_cluster_pain_related_mediane_originali.csv",
    "copertura_cluster_pain_related_patient_level.csv",
    "profili_pain_related_patient_level.csv",
    "valutazione_numero_cluster_pain_related.csv",
    "stabilita_cluster_pain_related.csv",
    "stabilita_ARI_cluster_pain_related.csv",
    "within_patient_pain_vs_rest_primary.csv",
    "within_patient_pain_vs_rest_sensitivity.csv",
    "controllo_durata_primary.csv",
    "controllo_durata_sensitivity.csv",
    "coppie_pain_rest_duration_matched_1s.csv",
    "within_patient_duration_matched_patient_level.csv",
    "duration_matched_primary_results.csv",
    "duration_matched_sensitivity_results.csv",
]

audit_files = []

for filename in EXPECTED_FILES:
    path = PAIN_DIR / filename
    audit_files.append({
        "file": filename,
        "exists": path.exists(),
        "size_kb": (
            round(path.stat().st_size / 1024, 2)
            if path.exists()
            else np.nan
        )
    })

audit_files_df = pd.DataFrame(audit_files)

display(audit_files_df)

print(
    "\nFile presenti:",
    int(audit_files_df["exists"].sum()),
    "/",
    len(audit_files_df)
)

assert audit_files_df["exists"].all()

print("\n✓ Audit completato: tutti i risultati del Notebook 15 sono presenti.")

,file,exists,size_kb
0,turni_pain_related_v3_definitivi.csv,True,617.89
1,riepilogo_pain_related_patient_level.csv,True,2.38
2,turni_pain_related_con_cluster_comportamentale.csv,True,627.75
3,profili_cluster_pain_related_standardizzati.csv,True,0.56
4,profili_cluster_pain_related_mediane_originali.csv,True,0.42
5,copertura_cluster_pain_related_patient_level.csv,True,0.27
6,profili_pain_related_patient_level.csv,True,2.59
7,valutazione_numero_cluster_pain_related.csv,True,0.40
8,stabilita_cluster_pain_related.csv,True,1.37
9,stabilita_ARI_cluster_pain_related.csv,True,0.98



File presenti: 18 / 18

✓ Audit completato: tutti i risultati del Notebook 15 sono presenti.
